# Key-Position Counterfactual Snapshot Profiler (Vi tri nao -> Quyet dinh nao Manh)

Notebook nay tu dong chup snapshot trang thai vat ly + RNG (Python/NumPy/Numba) tai **6 vi tri ban le trong 72h**:
1. `POS_1_MORNING_WARMUP`: Chuan bi mo cua / khoi dong AHU buoi sang
2. `POS_2_PEAK_COOLING`: Trua nang dong nguoi
3. `POS_3_MILD_SHOULDER`: Gio ngoai mat hon trong phong >= 1.5K (co hoi Economizer)
4. `POS_4_COMFORT_CRISIS`: Buoc Base dang bi phat comfort thuc te
5. `POS_5_EMPTY_DAY_START`: Ngay thuong vang nguoi sau 2h xac nhan
6. `POS_6_WEEKEND_WATER_HOLD`: Dem/cuoi tuan dai giu boiler 350K (>6h khong suoi)

Tu cung mot snapshot, he thong re nhanh song song `H = 24` buoc (`2 gio`) cho cac quyet dinh vat ly (`sat_+-2K`, `press_+-25%`, `recirc_eco`, `recirc_max`, `water_325K`, `water_310K`) + cac model Stage 1 / Stage 2 neu duoc attach (OOF Zero-Leak).


In [ ]:
%%writefile /kaggle/working/physics_foundation.py
"""Physics Foundation: TransferPolicy -> EmptyDayConsistentPolicy
Attribution: Daniel Wu, jazivxt (Facilities V42), penguin069 (Deep Setback)
Source: https://www.kaggle.com/code/penguin069/smart-buildings-empty-day-consistent-model
No neural weights. Only metadata and observations enter decisions.
"""
import math
import re
from collections import deque
import numpy as np


class TransferPolicy:
    'A transferable controller implementing the official two-method API.'
    name = 'adaptive_physics'
    DEFAULTS = {
        'comfort_margin': 0.12, 'precondition_hours': 1.0, 'recovery_steps': 1.5,
        'forgetting_factor': 0.997, 'model_residual_gain': 0.16, 'comfort_cost': 65000.0,
        'gas_cost_ratio': 0.30, 'hot_water_target': 345.0, 'standby_minimum': 282.0,
        'standby_maximum': 307.0, 'observation_delay': 1, 'response_scale': 0.75,
        'disturbance_rate': 0.35,
    }
    AIR_CAPACITY = 1.2 * 1006.0
    WATER_VALVE_CAPACITY = 4180.0 * 1000.0 * 0.108 / 3600.0
    DESIGN_PRESSURE = 2000.0
    MIN_PRESSURE = 200.0
    MIN_DAMPER = 0.10
    MAX_RECIRCULATION = 0.70
    MAX_DISCHARGE_K = 308.15

    def __init__(self, building_meta_data, action_spec=None, parameters=None):
        self.metadata = building_meta_data
        self.action_spec = action_spec
        self.parameters = dict(self.DEFAULTS)
        if parameters:
            self.parameters.update(parameters)
        self.n_continuous = int(building_meta_data['n_continuous_action'])
        self.n_discrete = int(building_meta_data['n_discrete_action'])
        self.field_names = [str(x) for x in building_meta_data['observation_field_names']]
        self.field_index = {name: i for i, name in enumerate(self.field_names)}
        self.obs_norm = building_meta_data.get('observation_normalization_metadata', {})
        self.action_norm = building_meta_data.get('action_normalization_metadata', {})
        self.temp_indices = list(building_meta_data['temp_indices'])
        self.pressure_indices = list(building_meta_data['pressure_indices'])
        self.recirculation_indices = list(building_meta_data['recirculation_indices'])
        self.water_indices = list(building_meta_data['water_indices'])
        self.num_ahus = len(self.temp_indices)
        zone_actions = building_meta_data['zone_action_indices']
        self.zone_ids = sorted(zone_actions, key=self._natural_key)
        self.n_zones = len(self.zone_ids)
        self.zone_number = {name: i for i, name in enumerate(self.zone_ids)}
        self.zone_obs = np.array([
            self.field_index.get(f'{z}_zone_air_temperature_sensor', -1) for z in self.zone_ids
        ], dtype=int)
        self.zone_low_obs = np.array([
            self.field_index.get(f'{z}_zone_air_heating_temperature_setpoint_sensor', -1) for z in self.zone_ids
        ], dtype=int)
        self.zone_high_obs = np.array([
            self.field_index.get(f'{z}_zone_air_cooling_temperature_setpoint_sensor', -1) for z in self.zone_ids
        ], dtype=int)
        self.zone_flow_obs = np.array([
            self.field_index.get(f'{z}_supply_air_flowrate_setpoint', -1) for z in self.zone_ids
        ], dtype=int)
        self.damper_indices = np.array([
            zone_actions[z]['supply_air_damper_percentage_command'] for z in self.zone_ids
        ], dtype=int)
        self.valve_indices = np.array([
            zone_actions[z]['reheat_valve_percentage_command'] for z in self.zone_ids
        ], dtype=int)
        assignments = building_meta_data.get('ahu_zone_assignments', building_meta_data.get('aha_zone_assignments', {}))
        self.ahu_ids = sorted(assignments, key=self._natural_key)
        self.groups = []
        self.zone_ahu = np.zeros(self.n_zones, dtype=int)
        self.outside_obs = []
        for i, ahu in enumerate(self.ahu_ids):
            group = np.array([
                self.zone_number[str(room).replace('room_', 'VAV_')]
                for room in assignments[ahu]
                if str(room).replace('room_', 'VAV_') in self.zone_number
            ], dtype=int)
            self.groups.append(group)
            self.zone_ahu[group] = i
            self.outside_obs.append(self._find_field(
                f'ahs_{ahu}_outside_air_temperature_sensor',
                f'{ahu}_outside_air_temperature_sensor',
                suffix='outside_air_temperature_sensor'))
        self.water_sensor = self._find_field(suffix='supply_water_temperature_sensor')
        self.comfort_now_index = self.field_index.get('comfort_mode_now')
        self.comfort_soon_index = self.field_index.get('comfort_mode_soon')
        self.occupancy_index = self.field_index.get('num_occupants')
        self.comfort_band = tuple(map(float, building_meta_data['comfort_temp_window']))
        self.eco_band = tuple(map(float, building_meta_data['eco_temp_window']))
        self.start_hour = float(building_meta_data.get('morning_start_hour', 6))
        self.end_hour = float(building_meta_data.get('evening_start_hour', 19))
        self.zone_area, self.zone_ua = self._geometry_priors(building_meta_data.get('zone_map'))
        effective_capacity = 3800.0 * self.zone_area + 21000.0 * np.sqrt(self.zone_area)
        self.initial_gain = np.clip(300000.0 / effective_capacity, 0.08, 2.5)
        self.reset()

    @staticmethod
    def _natural_key(value):
        return tuple(int(x) if x.isdigit() else x for x in re.split(r'(\d+)', str(value)))

    def _find_field(self, *exact, suffix=''):
        for name in exact:
            if name in self.field_index:
                return self.field_index[name]
        if suffix:
            for name, index in self.field_index.items():
                if name.endswith(suffix):
                    return index
        return None

    def _geometry_priors(self, zone_map):
        areas = np.full(self.n_zones, 55.0)
        ua = np.full(self.n_zones, 18.0)
        if zone_map is None:
            return areas, ua
        grid = np.asarray(zone_map)
        if grid.ndim != 2 or grid.size > 5000000:
            return areas, ua
        height, width = grid.shape
        inside = (grid == 0).ravel()
        visited = np.zeros(grid.size, dtype=bool)
        flat = grid.ravel()
        counts, external_edges = [], []
        for start in np.flatnonzero(inside):
            if visited[start]:
                continue
            stack = [int(start)]
            visited[start] = True
            count, edge = 0, 0
            while stack:
                p = stack.pop()
                count += 1
                row, col = divmod(p, width)
                for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                    nr, nc = row + dr, col + dc
                    if not (0 <= nr < height and 0 <= nc < width):
                        continue
                    q = nr * width + nc
                    if inside[q] and not visited[q]:
                        visited[q] = True
                        stack.append(q)
                    elif flat[q] == 2:
                        edge += 1
                    elif flat[q] == 1:
                        rr, cc = row + 2 * dr, col + 2 * dc
                        if 0 <= rr < height and 0 <= cc < width and grid[rr, cc] == 2:
                            edge += 1
            counts.append(count)
            external_edges.append(edge)
        if len(counts) != self.n_zones:
            return areas, ua
        for j, name in enumerate(self.zone_ids):
            match = re.search(r'(\d+)$', name)
            label = int(match.group(1)) - 1 if match else j
            if 0 <= label < len(counts):
                areas[j] = max(1.0, 0.04 * counts[label])
                ua[j] = max(0.5, external_edges[label] * 0.2 * 3.0 * 1.3)
        return areas, ua

    def reset(self):
        self.step_count = 0
        self.history = deque(maxlen=3)
        self.previous_observed = None
        self.slow_temperature = None
        self.residual = np.zeros(self.n_zones)
        self.model_error = np.full(self.n_zones, 0.08)
        self.theta = np.zeros((self.n_zones, 6))
        self.theta[:, 0] = self.initial_gain * self.zone_area * 2.0 / 1000.0
        self.theta[:, 1] = self.initial_gain * self.zone_ua * 20.0 / 1000.0
        self.theta[:, 2] = 0.18
        self.theta[:, 3] = 0.08
        self.theta[:, 4] = self.initial_gain * self.zone_area * 8.0 / 1000.0
        self.theta[:, 5] = self.initial_gain * self.parameters['response_scale']
        self.theta[:, :2] *= self.parameters['response_scale']
        self.theta[:, 4] *= self.parameters['response_scale']
        prior = np.array([0.12, 0.20, 0.15, 0.15, 0.16, 0.8])
        self.covariance = np.repeat(np.diag(prior)[None, :, :], self.n_zones, axis=0)
        self.last_heating_step = -1000
        self.last_action = None

    def _temperature(self, obs, index, kind, fallback):
        if index is None:
            return fallback
        value = float(obs[index])
        if 150.0 < value < 450.0:
            return value
        norm = self.obs_norm.get(kind, {})
        mean = float(norm.get('sample_mean', fallback))
        std = float(norm.get('sample_std', math.sqrt(norm.get('sample_variance', 1.0))))
        value = value * std + mean
        return value if np.isfinite(value) and 150.0 < value < 450.0 else fallback

    def _zone_temperatures(self, obs):
        norm = self.obs_norm.get('zone_air_temperature_sensor', {})
        mean = float(norm.get('sample_mean', 295.0))
        std = float(norm.get('sample_std', math.sqrt(norm.get('sample_variance', 25.0))))
        raw = obs[self.zone_obs].astype(float)
        value = np.where((raw > 150.0) & (raw < 450.0), raw, raw * std + mean)
        fallback = self.previous_observed if self.previous_observed is not None else mean
        return np.where(np.isfinite(value) & (value > 150.0) & (value < 450.0), value, fallback)

    def _hour(self, obs):
        sine = self.field_index.get('hod_sin_000')
        cosine = self.field_index.get('hod_cos_000')
        if sine is None or cosine is None:
            return (self.start_hour + self.step_count / 12.0) % 24.0
        return (math.atan2(float(obs[sine]), float(obs[cosine])) % (2.0 * math.pi)) * 12.0 / math.pi

    def _band(self, obs, temperature):
        now = bool(obs[self.comfort_now_index] > 0.5) if self.comfort_now_index is not None else (
            self.start_hour <= self._hour(obs) < self.end_hour)
        soon = bool(obs[self.comfort_soon_index] > 0.5) if self.comfort_soon_index is not None else now
        default = self.comfort_band if now else self.eco_band
        low = np.full(self.n_zones, default[0])
        high = np.full(self.n_zones, default[1])
        valid = (self.zone_low_obs >= 0) & (self.zone_high_obs >= 0)
        if np.any(valid):
            sensor_low = obs[self.zone_low_obs[valid]].astype(float)
            sensor_high = obs[self.zone_high_obs[valid]].astype(float)
            good = (sensor_low > 150.0) & (sensor_high < 450.0) & (sensor_high > sensor_low)
            indices = np.flatnonzero(valid)[good]
            low[indices] = sensor_low[good]
            high[indices] = sensor_high[good]
        if now:
            low = np.maximum(low, self.comfort_band[0])
            high = np.minimum(high, self.comfort_band[1])
        elif soon:
            hour = self._hour(obs)
            remaining = max(0.0, (self.start_hour - hour) % 24.0)
            progress = np.clip(1.0 - remaining / max(0.1, self.parameters['precondition_hours']), 0.0, 1.0)
            low = self.eco_band[0] + progress * (self.comfort_band[0] - self.eco_band[0]) + np.zeros(self.n_zones)
            high = self.eco_band[1] + progress * (self.comfort_band[1] - self.eco_band[1]) + np.zeros(self.n_zones)
        else:
            low[:] = self.parameters['standby_minimum']
            high[:] = self.parameters['standby_maximum']
        margin = self.parameters['comfort_margin'] if (now or soon) else 0.0
        low += margin
        high -= margin
        recovery = max(1.0, self.parameters['recovery_steps'])
        low = np.minimum(low, temperature + np.maximum((low - temperature) / recovery, 0.0))
        high = np.maximum(high, temperature - np.maximum((temperature - high) / recovery, 0.0))
        return low, high, now

    def _group_temperature(self, temperature):
        means = np.zeros(self.n_zones)
        for group in self.groups:
            if group.size:
                means[group] = np.average(temperature[group], weights=self.zone_area[group])
        return means

    def _features(self, temperature, outside, mode, slow):
        return np.column_stack((
            np.ones(self.n_zones),
            np.clip((outside - temperature) / 20.0, -5.0, 5.0),
            np.clip((slow - temperature) / 5.0, -5.0, 5.0),
            np.clip((self._group_temperature(temperature) - temperature) / 5.0, -5.0, 5.0),
            np.full(self.n_zones, float(mode)),
        ))

    def _learn(self, observed):
        delay = int(self.parameters['observation_delay'])
        if self.previous_observed is None or len(self.history) <= delay:
            return
        applied = list(self.history)[-(delay + 1)]
        old = self.previous_observed
        power = (applied['conductance'] * (applied['supply'] - observed) + applied['reheat']) / 1000.0
        features = self._features(old, applied['outside'], applied['mode'], applied['slow'])
        x = np.column_stack((features, power))
        delta = np.clip(observed - old, -15.0, 15.0)
        prediction = np.einsum('ij,ij->i', self.theta, x)
        remaining = delta - prediction
        rate = float(self.parameters['disturbance_rate'])
        self.model_error = 0.94 * self.model_error + 0.06 * np.minimum(np.abs(remaining - self.residual), 2.0)
        self.residual = np.clip((1.0 - rate) * self.residual + rate * remaining, -5.0, 5.0)

    def _free_drift(self, temperature, outside, mode):
        features = self._features(temperature, outside, mode, self.slow_temperature)
        return np.clip(np.einsum('ij,ij->i', self.theta[:, :5], features) + self.residual, -5.0, 5.0)

    def _native_action(self, kind, value, low, high):
        spec = self.action_norm.get(kind)
        if spec is None:
            spec = next((v for k, v in self.action_norm.items() if str(k).endswith(kind)), None)
        if spec:
            low = float(spec.get('min_native_value', low))
            high = float(spec.get('max_native_value', high))
            out_low = float(spec.get('min_normalized_value', -1.0))
            out_high = float(spec.get('max_normalized_value', 1.0))
        else:
            out_low, out_high = -1.0, 1.0
        return np.clip(out_low + (np.asarray(value) - low) * (out_high - out_low) / max(high - low, 1e-12), out_low, out_high)

    def _choose_ahu(self, group, temperature, free, low, high, max_flow, outside, water):
        t, d, lo, hi = temperature[group], free[group], low[group], high[group]
        response = self.theta[group, 5]
        capacity = max_flow[group]
        inside = float(np.average(t, weights=self.zone_area[group]))
        targets = np.array([285.0, 287.0, 289.0, 291.0, 293.0, 295.0, 297.0, 299.0, 302.0, 305.0])
        targets = np.concatenate((targets, [float(np.clip(np.min(hi) - 1.5, 285.0, 305.0))]))
        setpoint = np.repeat(targets, 2)
        recirc = np.full_like(setpoint, self.MAX_RECIRCULATION)
        if outside < inside - 0.5:
            recirc[1::2] = np.clip((targets - outside) / max(inside - outside, 1e-6), 0.0, self.MAX_RECIRCULATION)
        mixed = recirc * inside + (1.0 - recirc) * outside
        supply = np.minimum(mixed, setpoint)
        sa = supply[:, None]
        cool_need = np.maximum(t + d - hi, 0.0) * 1000.0 / response
        heat_need = np.maximum(lo - t - d, 0.0) * 1000.0 / response
        cooling_flow = cool_need[None, :] / (self.AIR_CAPACITY * np.maximum(hi[None, :] - sa, 0.05))
        heating_flow = heat_need[None, :] / (self.AIR_CAPACITY * np.maximum(self.MAX_DISCHARGE_K - lo[None, :], 0.5))
        maximum_water_heat = self.WATER_VALVE_CAPACITY * math.sqrt(20.0) * np.maximum(water - sa, 0.0)
        maximum_heating_flow = maximum_water_heat / (self.AIR_CAPACITY * np.maximum(self.MAX_DISCHARGE_K - sa, 0.1))
        heating_flow = np.minimum(heating_flow, maximum_heating_flow)
        needed_flow = np.maximum(cooling_flow, heating_flow)
        fraction = np.max(needed_flow / capacity[None, :], axis=1)
        pressure = np.clip(self.DESIGN_PRESSURE * fraction ** 2, self.MIN_PRESSURE, self.DESIGN_PRESSURE)
        pressure_ratio = np.sqrt(pressure / self.DESIGN_PRESSURE)
        dampers = np.clip(needed_flow / (capacity[None, :] * pressure_ratio[:, None]), self.MIN_DAMPER, 1.0)
        flow = dampers * capacity[None, :] * pressure_ratio[:, None]
        conductance = flow * self.AIR_CAPACITY
        water_gap = np.maximum(water - sa, 0.0)
        wanted_reheat = np.maximum(
            (lo[None, :] - t[None, :] - d[None, :]) * 1000.0 / response[None, :] + conductance * (lo[None, :] - sa), 0.0)
        reheat_limit = np.minimum(
            conductance * np.maximum(self.MAX_DISCHARGE_K - sa, 0.0),
            self.WATER_VALVE_CAPACITY * math.sqrt(20.0) * water_gap)
        reheat = np.minimum(wanted_reheat, reheat_limit)
        predicted = (t[None, :] + d[None, :] + response[None, :] * (conductance * sa + reheat) / 1000.0) / (
            1.0 + response[None, :] * conductance / 1000.0)
        violation = np.maximum(lo[None, :] - predicted, 0.0) + np.maximum(predicted - hi[None, :], 0.0)
        thermal_penalty = np.sum(violation + 2.0 * violation ** 2, axis=1)
        total_flow = flow.sum(axis=1)
        fan = total_flow * pressure * (2.0 - recirc) / 0.9
        chiller = total_flow * self.AIR_CAPACITY * np.maximum(mixed - supply, 0.0) / 3.5
        gas = reheat.sum(axis=1)
        objective = fan + chiller + self.parameters['gas_cost_ratio'] * gas
        objective += self.parameters['comfort_cost'] * thermal_penalty
        off_prediction = t + d
        off_violation = np.maximum(lo - off_prediction, 0.0) + np.maximum(off_prediction - hi, 0.0)
        off_objective = self.parameters['comfort_cost'] * np.sum(off_violation + 2.0 * off_violation ** 2)
        best = int(np.argmin(objective))
        if off_objective <= objective[best]:
            return {'run': 0, 'pressure': 0.0, 'setpoint': 305.0, 'recirc': 0.7,
                    'damper': np.full(group.size, self.MIN_DAMPER), 'conductance': np.zeros(group.size),
                    'supply': np.full(group.size, mixed[best]), 'reheat': np.zeros(group.size)}
        return {'run': 1, 'pressure': float(pressure[best]), 'setpoint': float(setpoint[best]),
                'recirc': float(recirc[best]), 'damper': dampers[best],
                'conductance': conductance[best], 'supply': np.full(group.size, supply[best]),
                'reheat': reheat[best]}

    def get_action(self, obs):
        obs = np.asarray(obs, dtype=float).reshape(-1)
        observed = self._zone_temperatures(obs)
        if self.slow_temperature is None:
            self.slow_temperature = observed.copy()
        self._learn(observed)
        outside_ahu = np.array([
            self._temperature(obs, index, 'outside_air_temperature_sensor', 290.0)
            for index in self.outside_obs
        ])
        outside = outside_ahu[self.zone_ahu]
        mode = bool(obs[self.comfort_now_index] > 0.5) if self.comfort_now_index is not None else True
        temperature = observed.copy()
        if self.history and int(self.parameters['observation_delay']) == 1:
            last = self.history[-1]
            drift = self._free_drift(observed, last['outside'], last['mode'])
            gain = self.theta[:, 5] / 1000.0
            temperature = (observed + drift + gain * (last['conductance'] * last['supply'] + last['reheat'])) / (
                1.0 + gain * last['conductance'])
            temperature = np.clip(temperature, observed - 8.0, observed + 8.0)
        low, high, mode = self._band(obs, temperature)
        free = self._free_drift(temperature, outside, mode)
        max_flow = np.full(self.n_zones, 2.0)
        valid = self.zone_flow_obs >= 0
        if np.any(valid):
            flow_reading = obs[self.zone_flow_obs[valid]]
            max_flow[valid] = np.where(np.isfinite(flow_reading) & (flow_reading > 0.0), flow_reading, 2.0)
        max_flow = np.clip(max_flow, 0.01, 100.0)
        water_observed = self._temperature(obs, self.water_sensor, 'supply_water_temperature_sensor', 340.0)
        heat_demand = np.any(temperature + free < low)
        if heat_demand:
            self.last_heating_step = self.step_count
        recent_heat = self.step_count - self.last_heating_step < 24
        water_target = self.parameters['hot_water_target'] if recent_heat else 310.0
        water_available = water_observed
        continuous = np.full(self.n_continuous, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_discrete, dtype=np.int32)
        conductance = np.zeros(self.n_zones)
        supply = outside.copy()
        reheat = np.zeros(self.n_zones)
        for i, group in enumerate(self.groups):
            if not group.size:
                continue
            choice = self._choose_ahu(group, temperature, free, low, high, max_flow,
                                      float(outside_ahu[i]), float(water_available))
            discrete[i] = choice['run']
            continuous[self.temp_indices[i]] = self._native_action(
                f'{self.ahu_ids[i]}_supply_air_temperature_setpoint', choice['setpoint'], 285.0, 305.0)
            continuous[self.pressure_indices[i]] = self._native_action(
                f'{self.ahu_ids[i]}_supply_air_static_pressure_setpoint', choice['pressure'], 0.0, 2000.0)
            continuous[self.recirculation_indices[i]] = self._native_action(
                f'{self.ahu_ids[i]}_recirculation_percentage_command', choice['recirc'], 0.0, 1.0)
            continuous[self.damper_indices[group]] = self._native_action(
                'supply_air_damper_percentage_command', choice['damper'], 0.0, 1.0)
            conductance[group] = choice['conductance']
            supply[group] = choice['supply']
            reheat[group] = choice['reheat']
        if np.any(reheat > 10.0):
            self.last_heating_step = self.step_count
            water_target = self.parameters['hot_water_target']
        valve_need = reheat / (self.WATER_VALVE_CAPACITY * np.maximum(water_available - supply, 1.0))
        pressure_water = float(np.clip(np.max(valve_need) ** 2 * 1.05, 0.0, 20.0))
        if pressure_water > 1e-8:
            valves = np.clip(valve_need / math.sqrt(pressure_water), 0.0, 1.0)
            discrete[-1] = 1
        else:
            valves = np.zeros(self.n_zones)
        continuous[self.valve_indices] = self._native_action('reheat_valve_percentage_command', valves, 0.0, 1.0)
        if self.water_indices:
            continuous[self.water_indices[0]] = self._native_action(
                'supply_water_temperature_setpoint', water_target, 310.0, 350.0)
        if len(self.pressure_indices) > self.num_ahus:
            continuous[self.pressure_indices[-1]] = self._native_action(
                'differential_pressure_setpoint', pressure_water, 0.0, 20.0)
        actual_reheat = np.minimum(
            self.WATER_VALVE_CAPACITY * math.sqrt(pressure_water) * valves * np.maximum(water_available - supply, 0.0),
            conductance * np.maximum(self.MAX_DISCHARGE_K - supply, 0.0))
        self.history.append({'conductance': conductance, 'supply': supply,
                             'reheat': actual_reheat, 'outside': outside.copy(),
                             'mode': mode, 'slow': self.slow_temperature.copy()})
        self.slow_temperature += (temperature - self.slow_temperature) / 48.0
        self.previous_observed = observed.copy()
        self.step_count += 1
        action = {'continuous_action': np.clip(np.nan_to_num(continuous, nan=-1.0), -1.0, 1.0).astype(np.float32),
                  'discrete_action': discrete.astype(np.int32)}
        self.last_action = action
        return action


class GuardedJointThermalPolicy(TransferPolicy):
    name = 'facilities_v40_joint_thermal'
    def _zone_temperatures(self, obs):
        value = super()._zone_temperatures(obs)
        previous = self.previous_observed if self.previous_observed is not None else np.mean(self.comfort_band)
        return np.where(np.isfinite(value) & (value >= 150.) & (value <= 400.), value, previous)


def bounded_rls(theta, covariance, x, target, lower, upper, forgetting=.997):
    px = np.einsum('nij,nj->ni', covariance, x)
    denominator = forgetting + np.einsum('ni,ni->n', x, px)
    gain = px / np.maximum(denominator[:, None], 1e-9)
    innovation = np.clip(target - np.einsum('ni,ni->n', theta, x), -2., 2.)
    fitted = np.clip(theta + gain * innovation[:, None], lower, upper)
    updated = (covariance - np.einsum('ni,nj->nij', gain, px)) / forgetting
    updated = .5 * (updated + updated.transpose(0, 2, 1))
    return fitted, updated


class IdentifiedJointThermalPolicy(GuardedJointThermalPolicy):
    name = 'facilities_v41_identified_thermal'
    def reset(self):
        super().reset()
        self._thermal_prior = self.theta[:, 5].copy()
    def _learn(self, observed):
        delay = int(self.parameters['observation_delay'])
        if self.previous_observed is None or len(self.history) <= delay:
            return
        applied = list(self.history)[-(delay + 1)]
        power = (applied['conductance'] * (applied['supply'] - observed) + applied['reheat']) / 1000.
        features = self._features(self.previous_observed, applied['outside'], applied['mode'], applied['slow'])
        x = np.column_stack((features, power))
        delta = observed - self.previous_observed
        valid = np.isfinite(x).all(axis=1) & np.isfinite(delta) & (np.abs(delta) <= 3.)
        lower = np.zeros_like(self.theta)
        upper = np.ones_like(self.theta) * np.array([2., 5., 2., 2., 2., 1.])
        lower[:, 0] = -2.
        lower[:, 5] = .5 * self._thermal_prior
        upper[:, 5] = 1.5 * self._thermal_prior
        if np.any(valid):
            fit, cov = bounded_rls(self.theta[valid], self.covariance[valid], x[valid],
                delta[valid] - self.residual[valid], lower[valid], upper[valid],
                float(self.parameters['forgetting_factor']))
            self.theta[valid], self.covariance[valid] = fit, cov
        remaining = np.clip(delta - np.einsum('ni,ni->n', self.theta, x), -5., 5.)
        self.model_error = .94 * self.model_error + .06 * np.minimum(np.abs(remaining - self.residual), 2.)
        rate = float(self.parameters['disturbance_rate'])
        self.residual = np.clip((1 - rate) * self.residual + rate * remaining, -5., 5.)


class ErrorReserveJointThermalPolicy(GuardedJointThermalPolicy):
    name = 'facilities_v42_error_reserve_thermal'
    def _band(self, obs, temperature):
        low, high, mode = super()._band(obs, temperature)
        if mode:
            extra = np.minimum(np.clip(.5 * (self.model_error - .08), 0., .25), .25 * np.maximum(high - low, 0.))
            low, high = low + extra, high - extra
        return low, high, mode


class IdentifiedReservePolicy(IdentifiedJointThermalPolicy, ErrorReserveJointThermalPolicy):
    name = 'identified_reserve'


class CausalIdentifiedReservePolicy(IdentifiedReservePolicy):
    name = 'causal_identified'
    def _learn(self, observed):
        delay = int(self.parameters['observation_delay'])
        if self.previous_observed is None or len(self.history) <= delay:
            return
        applied = list(self.history)[-(delay + 1)]
        old = self.previous_observed
        features = self._features(old, applied['outside'], applied['mode'], applied['slow'])
        drift = np.einsum('ij,ij->i', self.theta[:, :5], features) + self.residual
        gain = self.theta[:, 5] / 1000.0
        prediction = (old + drift + gain * (applied['conductance'] * applied['supply'] + applied['reheat'])) / (
            1.0 + gain * applied['conductance'])
        error = np.where(np.isfinite(observed - prediction), np.minimum(np.abs(observed - prediction), 2.0), 2.0)
        previous_error = self.model_error.copy()
        super()._learn(observed)
        self.model_error = 0.94 * previous_error + 0.06 * error


class WarmStandbyPolicy(CausalIdentifiedReservePolicy):
    name = 'causal_warm_standby'
    HOLD_STEPS = 36 * 12
    def _native_action(self, kind, value, low, high):
        if (kind == 'supply_water_temperature_setpoint'
                and self.step_count - self.last_heating_step < self.HOLD_STEPS):
            value = self.parameters['hot_water_target']
        return super()._native_action(kind, value, low, high)


class WeekendWaterHoldPolicy(WarmStandbyPolicy):
    name = 'warm_hold72'
    HOLD_STEPS = 72 * 12


class DeepFastRecoveryPolicy(WeekendWaterHoldPolicy):
    name = 'hold72_deep_fast_recovery'
    DEFAULTS = dict(WeekendWaterHoldPolicy.DEFAULTS, standby_minimum=274.0, recovery_steps=1.0)


class EmptyDayConsistentPolicy(DeepFastRecoveryPolicy):
    name = 'deep_empty_day_consistent_model'
    EMPTY_CONFIRMATION_HOURS = 2.0
    def reset(self):
        super().reset()
        self.empty_day_relaxed_steps = 0
    def _band(self, obs, temperature):
        low, high, schedule_mode = super()._band(obs, temperature)
        if not schedule_mode or self.occupancy_index is None:
            return low, high, schedule_mode
        occupants = float(obs[self.occupancy_index])
        elapsed = (self._hour(obs) - self.start_hour) % 24.0
        scheduled_duration = (self.end_hour - self.start_hour) % 24.0
        if (not np.isfinite(occupants) or occupants != 0.0
                or not self.EMPTY_CONFIRMATION_HOURS <= elapsed < scheduled_duration):
            return low, high, schedule_mode
        self.empty_day_relaxed_steps += 1
        return (np.full(self.n_zones, self.parameters['standby_minimum']),
                np.full(self.n_zones, self.parameters['standby_maximum']), schedule_mode)


In [ ]:
%%writefile /kaggle/working/warp_resip.py
"""Stochastic Warp RL (IQL-Warp) + Physics-Uncertainty ResiP Gate Module
Upgrades over v1:
1. RationalQuadraticWarp1D: Added hard upper slope bound (max_derivative=2.2)
   and exact closed-form analytical inverse() for Change-of-Variables log_prob.
2. DeepSetsWarpResiPActor: Stochastic Pre-Warp Gaussian N(x_base, sigma(s)^2)
   passed through invertible spline + Physics-Uncertainty-Gated ResiP (modulated
   by causal RLS model_error) + Antithetic MC inference (exact Identity at init).
3. DeepSetsValueNet & DeepSetsAHUTwinCritic: Enables Implicit Q-Learning (IQL)
   so Value/Critic NEVER query out-of-distribution actions during Bellman backups.
4. WarpResiPMixin & make_warp_policy(base_cls): Physics re-projection of VAV
   dampers and reheat valves preserving energy-mass conservation.
"""
import math
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

_HERE = str(Path(__file__).resolve().parent)
if _HERE not in sys.path:
    sys.path.insert(0, _HERE)

from physics_foundation import (
    CausalIdentifiedReservePolicy,
    EmptyDayConsistentPolicy,
    TransferPolicy,
)

ZONE_FEAT_DIM = 9
GLOBAL_FEAT_DIM = 10
AHU_ACT_DIM = 4  # [sat_norm, press_norm, recirc_norm, water_norm]


class RationalQuadraticWarp1D(nn.Module):
    """Bounded-slope Monotonic Rational-Quadratic Spline on [-bound, bound].

    Guarantees exact Identity at zero logits (W(x) == x) and Lipschitz slope bounds
    d in [min_derivative, max_derivative], with exact analytical forward & inverse.
    """
    def __init__(self, num_bins=8, bound=1.05, min_bin_width=0.05,
                 min_bin_height=0.05, min_derivative=0.40, max_derivative=2.20):
        super().__init__()
        self.K = num_bins
        self.B = float(bound)
        self.min_w = float(min_bin_width)
        self.min_h = float(min_bin_height)
        self.min_d = float(min_derivative)
        self.max_d = float(max_derivative)
        assert self.K * self.min_w < 2.0 * self.B
        assert self.K * self.min_h < 2.0 * self.B
        assert self.min_d < 1.0 < self.max_d
        t0 = (1.0 - self.min_d) / (self.max_d - self.min_d)
        self.c0 = math.log(t0 / (1.0 - t0))

    @property
    def num_params_per_dim(self):
        return 3 * self.K - 1

    def _compute_knots(self, raw_params, gate=None):
        K, B = self.K, self.B
        raw_w, raw_h, raw_d = torch.split(raw_params, [K, K, K - 1], dim=-1)
        if gate is not None:
            raw_w = gate * raw_w
            raw_h = gate * raw_h
            raw_d = gate * raw_d

        widths = self.min_w + (2.0 * B - K * self.min_w) * F.softmax(raw_w, dim=-1)
        heights = self.min_h + (2.0 * B - K * self.min_h) * F.softmax(raw_h, dim=-1)

        pad_lo = torch.full_like(widths[..., :1], -B)
        pad_hi = torch.full_like(widths[..., :1], B)
        cumwidths = torch.cat([pad_lo, torch.cumsum(widths, dim=-1)[..., :-1] - B, pad_hi], dim=-1)
        cumheights = torch.cat([pad_lo, torch.cumsum(heights, dim=-1)[..., :-1] - B, pad_hi], dim=-1)

        interior_d = self.min_d + (self.max_d - self.min_d) * torch.sigmoid(raw_d + self.c0)
        derivatives = F.pad(interior_d, (1, 1), value=1.0)
        return widths, heights, cumwidths, cumheights, derivatives

    def forward(self, x, raw_params, gate=None):
        K, B = self.K, self.B
        widths, heights, cumwidths, cumheights, derivatives = self._compute_knots(raw_params, gate)

        x_c = x.clamp(-B + 1e-6, B - 1e-6)
        bin_idx = ((x_c.unsqueeze(-1) >= cumwidths[..., :-1]).sum(dim=-1, keepdim=True) - 1).clamp(0, K - 1)

        xk = cumwidths.gather(-1, bin_idx).squeeze(-1)
        wk = widths.gather(-1, bin_idx).squeeze(-1)
        yk = cumheights.gather(-1, bin_idx).squeeze(-1)
        hk = heights.gather(-1, bin_idx).squeeze(-1)
        dk = derivatives.gather(-1, bin_idx).squeeze(-1)
        dk1 = derivatives.gather(-1, bin_idx + 1).squeeze(-1)

        sk = hk / wk.clamp(min=1e-8)
        theta = ((x_c - xk) / wk.clamp(min=1e-8)).clamp(0.0, 1.0)
        theta_1m = theta * (1.0 - theta)
        delta_d = dk1 + dk - 2.0 * sk

        num = hk * (sk * theta.pow(2) + dk * theta_1m)
        den = (sk + delta_d * theta_1m).clamp(min=1e-8)
        y_raw = yk + num / den

        grad_num = sk.pow(2) * (dk1 * theta.pow(2) + 2.0 * sk * theta_1m + dk * (1.0 - theta).pow(2))
        dydx = (grad_num / den.pow(2).clamp(min=1e-12)).clamp(min=1e-6)
        log_det_j = torch.log(dydx).sum(dim=-1)

        return y_raw.clamp(-1.0, 1.0), y_raw, log_det_j

    def inverse(self, y, raw_params, gate=None):
        """Exact closed-form inverse W^{-1}(y) and log|det d(W^{-1})/dy|."""
        K, B = self.K, self.B
        widths, heights, cumwidths, cumheights, derivatives = self._compute_knots(raw_params, gate)

        y_c = y.clamp(-B + 1e-6, B - 1e-6)
        bin_idx = ((y_c.unsqueeze(-1) >= cumheights[..., :-1]).sum(dim=-1, keepdim=True) - 1).clamp(0, K - 1)

        xk = cumwidths.gather(-1, bin_idx).squeeze(-1)
        wk = widths.gather(-1, bin_idx).squeeze(-1)
        yk = cumheights.gather(-1, bin_idx).squeeze(-1)
        hk = heights.gather(-1, bin_idx).squeeze(-1)
        dk = derivatives.gather(-1, bin_idx).squeeze(-1)
        dk1 = derivatives.gather(-1, bin_idx + 1).squeeze(-1)

        sk = hk / wk.clamp(min=1e-8)
        delta_d = dk1 + dk - 2.0 * sk
        dy = (y_c - yk).clamp(min=0.0)

        a = hk * (sk - dk) + dy * delta_d
        b = hk * dk - dy * delta_d
        c = -dy * sk
        disc = (b.pow(2) - 4.0 * a * c).clamp(min=1e-12)
        denom = (-b - torch.sqrt(disc)).clamp(max=-1e-12)
        theta = (2.0 * c / denom).clamp(0.0, 1.0)
        x_pre = xk + theta * wk

        theta_1m = theta * (1.0 - theta)
        den = (sk + delta_d * theta_1m).clamp(min=1e-8)
        grad_num = sk.pow(2) * (dk1 * theta.pow(2) + 2.0 * sk * theta_1m + dk * (1.0 - theta).pow(2))
        dydx = (grad_num / den.pow(2).clamp(min=1e-12)).clamp(min=1e-6)
        log_det_inv = -torch.log(dydx).sum(dim=-1)

        return x_pre, log_det_inv


class DeepSetsWarpResiPActor(nn.Module):
    """Stochastic DeepSets Spline-Warp Actor with Physics-Uncertainty Gate.

    Models pi(y | s) by sampling x_pre ~ N(x_base, sigma(s)^2) and warping
    through bounded RationalQuadraticWarp1D gated by RLS model_error.
    """
    def __init__(self, zone_dim=ZONE_FEAT_DIM, global_dim=GLOBAL_FEAT_DIM,
                 hidden=128, action_dim=AHU_ACT_DIM, num_bins=8,
                 min_std=0.02, max_std=0.18):
        super().__init__()
        self.action_dim = action_dim
        self.min_std = float(min_std)
        self.max_std = float(max_std)
        self.warp = RationalQuadraticWarp1D(num_bins=num_bins)
        n_spline = self.warp.num_params_per_dim

        self.zone_enc = nn.Sequential(
            nn.Linear(zone_dim, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(),
        )
        ctx_dim = hidden * 2 + global_dim + action_dim
        self.trunk = nn.Sequential(
            nn.Linear(ctx_dim, 256), nn.LayerNorm(256), nn.GELU(),
            nn.Linear(256, 256), nn.LayerNorm(256), nn.GELU(),
        )
        self.spline_head = nn.Linear(256, action_dim * n_spline)
        self.gate_head = nn.Linear(256, action_dim)
        self.std_head = nn.Linear(256, action_dim)

        nn.init.zeros_(self.spline_head.weight)
        nn.init.zeros_(self.spline_head.bias)
        nn.init.zeros_(self.gate_head.weight)
        nn.init.constant_(self.gate_head.bias, -2.0)
        nn.init.zeros_(self.std_head.weight)
        nn.init.constant_(self.std_head.bias, -1.2)  # initial std ~ 0.057

    def _encode(self, z_feats, z_mask, g_feats, x_base):
        h = self.zone_enc(z_feats) * z_mask
        denom = z_mask.sum(dim=1).clamp(min=1.0)
        z_mean = h.sum(dim=1) / denom
        z_max = h.masked_fill(z_mask.expand_as(h) < 0.5, -1e9).max(dim=1).values
        z_max = torch.where(denom > 0, z_max, torch.zeros_like(z_max))

        # Physics-Uncertainty Gate Modulation from causal RLS model_error (col 5)
        me_mean = (z_feats[:, :, 5:6] * z_mask).sum(dim=1) / denom
        unc_scale = torch.clamp((me_mean - 0.04) / 0.16, min=0.20, max=1.0)

        ctx = torch.cat([z_mean, z_max, g_feats, x_base], dim=-1)
        feat = self.trunk(ctx)

        raw_spline = self.spline_head(feat).view(-1, self.action_dim, self.warp.num_params_per_dim)
        gate = unc_scale * torch.sigmoid(self.gate_head(feat))
        std = self.min_std + (self.max_std - self.min_std) * torch.sigmoid(self.std_head(feat))
        return raw_spline, gate, std

    def forward(self, z_feats, z_mask, g_feats, x_base, n_antithetic=4):
        """Antithetic MC expectation over warped stochastic distribution.

        At identity init (W == Identity), E[W(x_base +- eps)] == x_base identically!
        """
        raw_spline, gate, std = self._encode(z_feats, z_mask, g_feats, x_base)
        if n_antithetic <= 0:
            x_warped, _, log_det = self.warp(x_base, raw_spline, gate=gate.unsqueeze(-1))
            return x_warped, gate, log_det, std

        # Symmetric quadrature nodes over N(0, std(s)^2) capped inside [-B, B]
        B_sz = x_base.shape[0]
        nodes = torch.tensor([0.3, 0.8, 1.5], device=x_base.device, dtype=x_base.dtype).view(1, 3, 1)
        headroom = (self.warp.B - 1e-4 - x_base.abs()).clamp(min=0.0).unsqueeze(1)  # [B, 1, D]
        deltas = torch.minimum(std.unsqueeze(1) * nodes, headroom)                   # [B, 3, D]
        x_pre = torch.cat([x_base.unsqueeze(1) + deltas, x_base.unsqueeze(1) - deltas], dim=1)  # [B, 6, D]
        rs_exp = raw_spline.unsqueeze(1).expand(B_sz, 6, -1, -1)
        gt_exp = gate.unsqueeze(1).unsqueeze(-1).expand(B_sz, 6, -1, -1)

        _, y_raw_mc, log_det_mc = self.warp(x_pre, rs_exp, gate=gt_exp)
        x_expected = y_raw_mc.mean(dim=1).clamp(-1.0, 1.0)
        log_det_mean = log_det_mc.mean(dim=1)
        return x_expected, gate, log_det_mean, std

    def sample(self, z_feats, z_mask, g_feats, x_base):
        """Stochastic sample and exact change-of-variables log_prob."""
        raw_spline, gate, std = self._encode(z_feats, z_mask, g_feats, x_base)
        eps = torch.randn_like(x_base)
        x_pre = x_base + std * eps
        y_clamped, y_raw, log_det_j = self.warp(x_pre, raw_spline, gate=gate.unsqueeze(-1))
        log_p0 = (-0.5 * (eps.pow(2) + 2.0 * torch.log(std) + math.log(2.0 * math.pi))).sum(dim=-1)
        log_prob = log_p0 - log_det_j
        return y_clamped, log_prob, gate, log_det_j, std

    def log_prob(self, z_feats, z_mask, g_feats, x_base, y_data):
        """Exact log pi(y_data | s) via closed-form spline inverse for IQL-AWR."""
        raw_spline, gate, std = self._encode(z_feats, z_mask, g_feats, x_base)
        x_pre, log_det_inv = self.warp.inverse(y_data, raw_spline, gate=gate.unsqueeze(-1))
        z_norm = (x_pre - x_base) / std
        log_p0 = (-0.5 * (z_norm.pow(2) + 2.0 * torch.log(std) + math.log(2.0 * math.pi))).sum(dim=-1)
        log_pi = log_p0 + log_det_inv
        return log_pi, gate, log_det_inv, std


class DeepSetsValueNet(nn.Module):
    """State Value Network V(s) for Implicit Q-Learning (Expectile Regression)."""
    def __init__(self, zone_dim=ZONE_FEAT_DIM, global_dim=GLOBAL_FEAT_DIM, hidden=128):
        super().__init__()
        self.zone_enc = nn.Sequential(
            nn.Linear(zone_dim, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(),
        )
        self.v_net = nn.Sequential(
            nn.Linear(hidden * 2 + global_dim, 256), nn.LayerNorm(256), nn.GELU(),
            nn.Linear(256, 256), nn.LayerNorm(256), nn.GELU(),
            nn.Linear(256, 1),
        )

    def forward(self, z_feats, z_mask, g_feats):
        h = self.zone_enc(z_feats) * z_mask
        denom = z_mask.sum(dim=1).clamp(min=1.0)
        z_mean = h.sum(dim=1) / denom
        z_max = h.masked_fill(z_mask.expand_as(h) < 0.5, -1e9).max(dim=1).values
        z_max = torch.where(denom > 0, z_max, torch.zeros_like(z_max))
        return self.v_net(torch.cat([z_mean, z_max, g_feats], dim=-1))


class _WarpCriticHead(nn.Module):
    def __init__(self, zone_dim, global_dim, hidden, action_dim):
        super().__init__()
        self.zone_enc = nn.Sequential(
            nn.Linear(zone_dim + action_dim, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(),
        )
        self.q_net = nn.Sequential(
            nn.Linear(hidden * 2 + global_dim + action_dim, 256), nn.LayerNorm(256), nn.GELU(),
            nn.Linear(256, 256), nn.LayerNorm(256), nn.GELU(),
            nn.Linear(256, 1),
        )

    def forward(self, z_feats, z_mask, g_feats, x_eff):
        x_exp = x_eff.unsqueeze(1).expand(-1, z_feats.shape[1], -1)
        h = self.zone_enc(torch.cat([z_feats, x_exp], dim=-1)) * z_mask
        denom = z_mask.sum(dim=1).clamp(min=1.0)
        z_mean = h.sum(dim=1) / denom
        z_max = h.masked_fill(z_mask.expand_as(h) < 0.5, -1e9).max(dim=1).values
        z_max = torch.where(denom > 0, z_max, torch.zeros_like(z_max))
        return self.q_net(torch.cat([z_mean, z_max, g_feats, x_eff], dim=-1))


class DeepSetsAHUTwinCritic(nn.Module):
    """Building-level Twin Critic receiving 4D warped action x_eff in [-1, 1]^4."""
    def __init__(self, zone_dim=ZONE_FEAT_DIM, global_dim=GLOBAL_FEAT_DIM,
                 hidden=128, action_dim=AHU_ACT_DIM):
        super().__init__()
        self.q1 = _WarpCriticHead(zone_dim, global_dim, hidden, action_dim)
        self.q2 = _WarpCriticHead(zone_dim, global_dim, hidden, action_dim)

    def forward(self, z_feats, z_mask, g_feats, x_eff):
        return self.q1(z_feats, z_mask, g_feats, x_eff), self.q2(z_feats, z_mask, g_feats, x_eff)

    def Q1(self, z_feats, z_mask, g_feats, x_eff):
        return self.q1(z_feats, z_mask, g_feats, x_eff)


def extract_warp_state(policy, obs_arr, temperature, low, high, mode, oat_ema):
    outside_ahu = np.array([
        policy._temperature(obs_arr, idx, 'outside_air_temperature_sensor', 290.0)
        for idx in policy.outside_obs
    ])
    outside = outside_ahu[policy.zone_ahu]
    mean_oat = float(np.mean(outside_ahu))
    oat_ema = mean_oat if oat_ema is None else (0.98 * oat_ema + 0.02 * mean_oat)

    free = policy._free_drift(temperature, outside, mode)
    z_feats = np.column_stack([
        np.clip((temperature - low) / 2.0, -5.0, 5.0),
        np.clip((high - temperature) / 2.0, -5.0, 5.0),
        np.clip(free / 2.0, -3.0, 3.0),
        np.clip((outside - temperature) / 20.0, -5.0, 5.0),
        np.clip((policy.slow_temperature - temperature) / 5.0, -5.0, 5.0),
        np.clip(policy.model_error, 0.0, 2.0),
        np.clip(policy.residual / 2.0, -2.0, 2.0),
        np.log(np.clip(policy.theta[:, 5], 1e-3, 10.0)),
        np.log1p(policy.zone_area) / 5.0,
    ]).astype(np.float32)
    z_feats = np.nan_to_num(z_feats, nan=0.0, posinf=5.0, neginf=-5.0)

    hour = policy._hour(obs_arr)
    sine_idx = policy.field_index.get('hod_sin_000')
    cos_idx = policy.field_index.get('hod_cos_000')
    hod_sin = float(obs_arr[sine_idx]) if sine_idx is not None else math.sin(hour * math.pi / 12.0)
    hod_cos = float(obs_arr[cos_idx]) if cos_idx is not None else math.cos(hour * math.pi / 12.0)

    c_now = float(obs_arr[policy.comfort_now_index] > 0.5) if policy.comfort_now_index is not None else float(mode)
    c_soon = float(obs_arr[policy.comfort_soon_index] > 0.5) if policy.comfort_soon_index is not None else c_now
    hours_to_start = ((policy.start_hour - hour) % 24.0) / 24.0
    occ_raw = float(obs_arr[policy.occupancy_index]) if policy.occupancy_index is not None else 0.0
    occ_norm = math.log1p(max(occ_raw, 0.0)) / 5.0 if np.isfinite(occ_raw) else 0.0
    water_obs = policy._temperature(obs_arr, policy.water_sensor, 'supply_water_temperature_sensor', 340.0)
    heat_recency = math.exp(-max(0, policy.step_count - policy.last_heating_step) / 144.0)

    g_feats = np.array([
        hod_sin, hod_cos,
        np.clip((mean_oat - 290.0) / 15.0, -3.0, 3.0),
        np.clip((mean_oat - oat_ema) / 5.0, -3.0, 3.0),
        c_now, c_soon, hours_to_start, occ_norm,
        np.clip((water_obs - 330.0) / 20.0, -3.0, 3.0),
        heat_recency,
    ], dtype=np.float32)
    g_feats = np.nan_to_num(g_feats, nan=0.0, posinf=3.0, neginf=-3.0)
    return z_feats, g_feats, oat_ema, outside_ahu, outside, free


class WarpResiPMixin:
    """Mixin layering Stochastic Warp RL + Physics-Uncertainty Gate + Physics Safety Shield."""
    def __init__(self, building_meta_data, action_spec=None, parameters=None,
                 actors=None, action_modifier=None, dev=None, smooth_alpha=0.5,
                 comfort_shield=True, target_water_only=False):
        self._actors = actors if isinstance(actors, (list, tuple)) else ([actors] if actors is not None else [])
        self._action_modifier = action_modifier
        self._dev = dev or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self._smooth_alpha = float(smooth_alpha)
        self._comfort_shield = bool(comfort_shield)
        self._target_water_only = bool(target_water_only)
        super().__init__(building_meta_data, action_spec, parameters)

    def reset(self):
        super().reset()
        self._oat_ema = None
        self._last_transition = None
        self._last_gate = np.zeros(AHU_ACT_DIM, dtype=np.float32)
        self._last_thermal_sample = None
        self._prev_delta_x = np.zeros(AHU_ACT_DIM, dtype=np.float32)
        self._shield_checks = 0
        self._shield_hits = 0

    def _learn(self, observed):
        delay = int(self.parameters['observation_delay'])
        self._last_thermal_sample = None
        if self.previous_observed is not None and len(self.history) > delay:
            applied = list(self.history)[-(delay + 1)]
            old = self.previous_observed
            features = self._features(old, applied['outside'], applied['mode'], applied['slow'])
            drift = np.einsum('ij,ij->i', self.theta[:, :5], features) + self.residual
            gain = self.theta[:, 5] / 1000.0
            pred_next = (old + drift + gain * (applied['conductance'] * applied['supply'] + applied['reheat'])) / (
                1.0 + gain * applied['conductance'])
            power = (applied['conductance'] * (applied['supply'] - observed) + applied['reheat']) / 1000.0
            self._last_thermal_sample = {
                'old': old.copy(),
                'pred_next': pred_next.copy(),
                'observed': observed.copy(),
                'features': features.copy(),
                'power': power.copy(),
                'model_error': self.model_error.copy(),
                'residual': self.residual.copy(),
                'theta_gain': self.theta[:, 5].copy(),
                'zone_area': self.zone_area.copy(),
            }
        super()._learn(observed)

    def _ahu_step_metrics(self, group, temperature, free, low, high, outside, choice):
        t, d, lo, hi = temperature[group], free[group], low[group], high[group]
        response = self.theta[group, 5]
        inside = float(np.average(t, weights=self.zone_area[group]))
        supply = choice['supply']
        conductance = choice['conductance']
        reheat = choice['reheat']
        predicted = (t + d + response * (conductance * supply + reheat) / 1000.0) / (
            1.0 + response * conductance / 1000.0)
        violation = np.maximum(lo - predicted, 0.0) + np.maximum(predicted - hi, 0.0)
        thermal_penalty = float(np.sum(violation + 2.0 * violation ** 2))
        if choice['run'] == 0:
            obj = float(self.parameters['comfort_cost']) * thermal_penalty
        else:
            flow = conductance / self.AIR_CAPACITY
            mixed = choice['recirc'] * inside + (1.0 - choice['recirc']) * outside
            fan = float(flow.sum()) * choice['pressure'] * (2.0 - choice['recirc']) / 0.9
            chiller = float((flow * self.AIR_CAPACITY * np.maximum(mixed - supply, 0.0)).sum()) / 3.5
            gas = float(reheat.sum())
            obj = fan + chiller + float(self.parameters['gas_cost_ratio']) * gas + float(self.parameters['comfort_cost']) * thermal_penalty
        return thermal_penalty, float(obj)

    def _reproject_ahu(self, group, temperature, free, low, high, max_flow, outside, water,
                       setpoint_w, pressure_w, recirc_w):
        t, d, lo, hi = temperature[group], free[group], low[group], high[group]
        response = self.theta[group, 5]
        capacity = max_flow[group]
        inside = float(np.average(t, weights=self.zone_area[group]))

        mixed = recirc_w * inside + (1.0 - recirc_w) * outside
        supply = min(mixed, setpoint_w)

        cool_need = np.maximum(t + d - hi, 0.0) * 1000.0 / response
        heat_need = np.maximum(lo - t - d, 0.0) * 1000.0 / response
        cooling_flow = cool_need / (self.AIR_CAPACITY * np.maximum(hi - supply, 0.05))
        heating_flow = heat_need / (self.AIR_CAPACITY * np.maximum(self.MAX_DISCHARGE_K - lo, 0.5))

        maximum_water_heat = self.WATER_VALVE_CAPACITY * math.sqrt(20.0) * max(water - supply, 0.0)
        maximum_heating_flow = maximum_water_heat / (self.AIR_CAPACITY * max(self.MAX_DISCHARGE_K - supply, 0.1))
        heating_flow = np.minimum(heating_flow, maximum_heating_flow)
        needed_flow = np.maximum(cooling_flow, heating_flow)

        pressure_ratio = math.sqrt(pressure_w / self.DESIGN_PRESSURE)
        dampers = np.clip(needed_flow / (capacity * pressure_ratio), self.MIN_DAMPER, 1.0)
        flow = dampers * capacity * pressure_ratio
        conductance = flow * self.AIR_CAPACITY

        water_gap = max(water - supply, 0.0)
        wanted_reheat = np.maximum(
            (lo - t - d) * 1000.0 / response + conductance * (lo - supply), 0.0)
        reheat_limit = np.minimum(
            conductance * max(self.MAX_DISCHARGE_K - supply, 0.0),
            self.WATER_VALVE_CAPACITY * math.sqrt(20.0) * water_gap)
        reheat = np.minimum(wanted_reheat, reheat_limit)

        return {
            'run': 1, 'pressure': float(pressure_w), 'setpoint': float(setpoint_w),
            'recirc': float(recirc_w), 'damper': dampers, 'conductance': conductance,
            'supply': np.full(group.size, supply), 'reheat': reheat,
        }

    def get_action(self, obs):
        obs_arr = np.asarray(obs, dtype=float).reshape(-1)
        observed = self._zone_temperatures(obs_arr)
        if self.slow_temperature is None:
            self.slow_temperature = observed.copy()
        self._learn(observed)

        outside_ahu = np.array([
            self._temperature(obs_arr, index, 'outside_air_temperature_sensor', 290.0)
            for index in self.outside_obs
        ])
        outside = outside_ahu[self.zone_ahu]
        mode = bool(obs_arr[self.comfort_now_index] > 0.5) if self.comfort_now_index is not None else True

        temperature = observed.copy()
        if self.history and int(self.parameters['observation_delay']) == 1:
            last = self.history[-1]
            drift = self._free_drift(observed, last['outside'], last['mode'])
            gain = self.theta[:, 5] / 1000.0
            temperature = (observed + drift + gain * (last['conductance'] * last['supply'] + last['reheat'])) / (
                1.0 + gain * last['conductance'])
            temperature = np.clip(temperature, observed - 8.0, observed + 8.0)

        low, high, mode = self._band(obs_arr, temperature)
        z_f, g_f, self._oat_ema, _, _, free = extract_warp_state(
            self, obs_arr, temperature, low, high, mode, self._oat_ema)

        max_flow = np.full(self.n_zones, 2.0)
        valid = self.zone_flow_obs >= 0
        if np.any(valid):
            flow_reading = obs_arr[self.zone_flow_obs[valid]]
            max_flow[valid] = np.where(np.isfinite(flow_reading) & (flow_reading > 0.0), flow_reading, 2.0)
        max_flow = np.clip(max_flow, 0.01, 100.0)

        water_observed = self._temperature(obs_arr, self.water_sensor, 'supply_water_temperature_sensor', 340.0)
        if np.any(temperature + free < low):
            self.last_heating_step = self.step_count
        recent_heat = self.step_count - self.last_heating_step < 24
        water_available = water_observed

        base_choices = []
        any_reheat_10 = False
        for i, group in enumerate(self.groups):
            if not group.size:
                base_choices.append(None)
                continue
            c = self._choose_ahu(group, temperature, free, low, high, max_flow,
                                 float(outside_ahu[i]), float(water_available))
            base_choices.append(c)
            if np.any(c['reheat'] > 10.0):
                any_reheat_10 = True

        if any_reheat_10:
            self.last_heating_step = self.step_count
        hold_steps = getattr(self, 'HOLD_STEPS', 24)
        hold_active = self.step_count - self.last_heating_step < hold_steps
        base_water_target = float(self.parameters['hot_water_target']) if (recent_heat or any_reheat_10 or hold_active) else 310.0

        active_indices = [i for i, c in enumerate(base_choices) if c is not None and c['run'] == 1]
        if active_indices:
            weights = np.array([self.zone_area[self.groups[i]].sum() for i in active_indices], dtype=float)
            weights /= max(weights.sum(), 1e-8)
            sat_b = sum(w * base_choices[i]['setpoint'] for w, i in zip(weights, active_indices))
            press_b = sum(w * base_choices[i]['pressure'] for w, i in zip(weights, active_indices))
            recirc_b = sum(w * base_choices[i]['recirc'] for w, i in zip(weights, active_indices))
        else:
            sat_b, press_b, recirc_b = 305.0, 0.0, 0.7

        x_base = np.array([
            np.clip((sat_b - 295.0) / 10.0, -1.0, 1.0),
            np.clip((press_b - 1000.0) / 1000.0, -1.0, 1.0),
            np.clip((recirc_b - 0.5) / 0.5, -1.0, 1.0),
            np.clip((base_water_target - 330.0) / 20.0, -1.0, 1.0),
        ], dtype=np.float32)

        gate_val = np.zeros(AHU_ACT_DIM, dtype=np.float32)
        if self._action_modifier is not None:
            x_eff = self._action_modifier(self.step_count, z_f, g_f, x_base)
        elif self._actors:
            zt = torch.tensor(z_f, dtype=torch.float32, device=self._dev).unsqueeze(0)
            zm = torch.ones((1, z_f.shape[0], 1), dtype=torch.float32, device=self._dev)
            gt = torch.tensor(g_f, dtype=torch.float32, device=self._dev).unsqueeze(0)
            xb = torch.tensor(x_base, dtype=torch.float32, device=self._dev).unsqueeze(0)
            with torch.no_grad():
                outs = [net(zt, zm, gt, xb) for net in self._actors]
            x_raw = np.mean([o[0].squeeze(0).cpu().numpy() for o in outs], axis=0).astype(np.float32)
            gate_val = np.mean([o[1].squeeze(0).cpu().numpy() for o in outs], axis=0).astype(np.float32)
            # Temporal delta smoothing (prevents 5-minute actuator hunting)
            raw_dx = x_raw - x_base
            if not np.allclose(raw_dx, 0.0, atol=1e-7):
                sm_dx = self._smooth_alpha * raw_dx + (1.0 - self._smooth_alpha) * self._prev_delta_x
                self._prev_delta_x = sm_dx
                x_eff = np.clip(x_base + sm_dx, -1.0, 1.0).astype(np.float32)
            else:
                self._prev_delta_x = np.zeros(AHU_ACT_DIM, dtype=np.float32)
                x_eff = x_base.copy()
        else:
            x_eff = x_base.copy()

        if self._target_water_only:
            x_eff = np.array([x_base[0], x_base[1], x_base[2], x_eff[3]], dtype=np.float32)

        delta_x = x_eff - x_base
        if np.allclose(delta_x, 0.0, atol=1e-7):
            water_target = base_water_target
        else:
            water_target = float(np.clip(330.0 + 20.0 * float(x_eff[3]), 310.0, 350.0))
            if self._comfort_shield and self._action_modifier is None and (any_reheat_10 or np.any(temperature + free < low)):
                water_target = max(water_target, base_water_target)

        continuous = np.full(self.n_continuous, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_discrete, dtype=np.int32)
        conductance = np.zeros(self.n_zones)
        supply = outside.copy()
        reheat = np.zeros(self.n_zones)

        for i, group in enumerate(self.groups):
            c = base_choices[i]
            if c is None:
                continue
            if c['run'] == 1 and not self._target_water_only and not np.allclose(delta_x[:3], 0.0, atol=1e-7):
                sat_w = float(np.clip(c['setpoint'] + 10.0 * float(delta_x[0]), 285.0, 305.0))
                press_w = float(np.clip(c['pressure'] + 1000.0 * float(delta_x[1]), self.MIN_PRESSURE, self.DESIGN_PRESSURE))
                recirc_w = float(np.clip(c['recirc'] + 0.5 * float(delta_x[2]), 0.0, self.MAX_RECIRCULATION))
                c_w = self._reproject_ahu(
                    group, temperature, free, low, high, max_flow,
                    float(outside_ahu[i]), float(water_available), sat_w, press_w, recirc_w)
                if self._comfort_shield and self._action_modifier is None:
                    self._shield_checks += 1
                    pen_b, _ = self._ahu_step_metrics(group, temperature, free, low, high, float(outside_ahu[i]), c)
                    pen_w, obj_w = self._ahu_step_metrics(group, temperature, free, low, high, float(outside_ahu[i]), c_w)
                    if (not np.isfinite(obj_w)) or (pen_w > pen_b + 1e-6):
                        self._shield_hits += 1
                    else:
                        c = c_w
                else:
                    c = c_w

            discrete[i] = c['run']
            continuous[self.temp_indices[i]] = TransferPolicy._native_action(
                self, f'{self.ahu_ids[i]}_supply_air_temperature_setpoint', c['setpoint'], 285.0, 305.0)
            continuous[self.pressure_indices[i]] = TransferPolicy._native_action(
                self, f'{self.ahu_ids[i]}_supply_air_static_pressure_setpoint', c['pressure'], 0.0, 2000.0)
            continuous[self.recirculation_indices[i]] = TransferPolicy._native_action(
                self, f'{self.ahu_ids[i]}_recirculation_percentage_command', c['recirc'], 0.0, 1.0)
            continuous[self.damper_indices[group]] = TransferPolicy._native_action(
                self, 'supply_air_damper_percentage_command', c['damper'], 0.0, 1.0)
            conductance[group] = c['conductance']
            supply[group] = c['supply']
            reheat[group] = c['reheat']

        if np.any(reheat > 10.0):
            self.last_heating_step = self.step_count

        valve_need = reheat / (self.WATER_VALVE_CAPACITY * np.maximum(water_available - supply, 1.0))
        pressure_water = float(np.clip(np.max(valve_need) ** 2 * 1.05, 0.0, 20.0))
        if pressure_water > 1e-8:
            valves = np.clip(valve_need / math.sqrt(pressure_water), 0.0, 1.0)
            discrete[-1] = 1
        else:
            valves = np.zeros(self.n_zones)

        continuous[self.valve_indices] = TransferPolicy._native_action(
            self, 'reheat_valve_percentage_command', valves, 0.0, 1.0)
        if self.water_indices:
            continuous[self.water_indices[0]] = TransferPolicy._native_action(
                self, 'supply_water_temperature_setpoint', water_target, 310.0, 350.0)
        if len(self.pressure_indices) > self.num_ahus:
            continuous[self.pressure_indices[-1]] = TransferPolicy._native_action(
                self, 'differential_pressure_setpoint', pressure_water, 0.0, 20.0)

        actual_reheat = np.minimum(
            self.WATER_VALVE_CAPACITY * math.sqrt(pressure_water) * valves * np.maximum(water_available - supply, 0.0),
            conductance * np.maximum(self.MAX_DISCHARGE_K - supply, 0.0))
        self.history.append({'conductance': conductance, 'supply': supply,
                             'reheat': actual_reheat, 'outside': outside.copy(),
                             'mode': mode, 'slow': self.slow_temperature.copy()})
        self.slow_temperature += (temperature - self.slow_temperature) / 48.0
        self.previous_observed = observed.copy()
        self.step_count += 1
        self._last_transition = (z_f, g_f, x_base, x_eff)
        self._last_gate = gate_val

        action = {'continuous_action': np.clip(np.nan_to_num(continuous, nan=-1.0), -1.0, 1.0).astype(np.float32),
                  'discrete_action': discrete.astype(np.int32)}
        self.last_action = action
        return action


def make_warp_policy(base_cls):
    class _ConfiguredWarpResiPPolicy(WarpResiPMixin, base_cls):
        name = f'warp_resip_{getattr(base_cls, "name", "base")}'
    return _ConfiguredWarpResiPPolicy


class WarpResiPPolicy(WarpResiPMixin, EmptyDayConsistentPolicy):
    name = 'warp_resip_empty_day'


class WarpResiPCausalPolicy(WarpResiPMixin, CausalIdentifiedReservePolicy):
    name = 'warp_resip_causal_identified'


def verify_warp_resip_module():
    """Sanity check: identity init, closed-form invertibility, log_prob & gradients."""
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    actor = DeepSetsWarpResiPActor().to(dev)
    critic = DeepSetsAHUTwinCritic().to(dev)
    vnet = DeepSetsValueNet().to(dev)

    z = torch.randn(4, 12, ZONE_FEAT_DIM, device=dev)
    z[:, :, 5] = 0.25  # realistic RLS model_error
    m = torch.ones(4, 12, 1, device=dev)
    g = torch.randn(4, GLOBAL_FEAT_DIM, device=dev)
    xb = torch.linspace(-0.9, 0.9, 16, device=dev).view(4, 4)

    # 1. Exact Identity check with Antithetic MC
    xw, gate, log_det, std = actor(z, m, g, xb, n_antithetic=4)
    max_id_err = float((xw - xb).abs().max().item())
    max_ld_err = float(log_det.abs().max().item())

    # 2. Exact closed-form inverse check on nontrivial random spline params
    raw_test = torch.randn(4, 4, actor.warp.num_params_per_dim, device=dev) * 0.8
    _, y_raw, ld_fwd = actor.warp(xb, raw_test)
    x_rec, ld_inv = actor.warp.inverse(y_raw, raw_test)
    max_inv_err = float((x_rec - xb).abs().max().item())
    max_jac_err = float((ld_fwd + ld_inv).abs().max().item())

    # 3. Gradient flow check through IQL log_prob + CAPS
    log_pi, _, _, _ = actor.log_prob(z, m, g, xb, xb + 0.05)
    v_val = vnet(z, m, g)
    q1, _ = critic(z, m, g, xw)
    loss = -(log_pi.mean()) + (v_val - q1.detach()).pow(2).mean()
    loss.backward()
    has_grad = all(p.grad is not None and torch.isfinite(p.grad).all()
                   for p in actor.parameters() if p.requires_grad)
    return {
        'max_identity_err': max_id_err,
        'max_logdet_err': max_ld_err,
        'max_inv_err': max_inv_err,
        'max_jac_err': max_jac_err,
        'grad_ok': has_grad,
    }


In [ ]:
%%writefile /kaggle/working/thermal_residual_net.py
"""Stage 1: Physics-Informed Neural Thermal Residual Net (_free_drift upgrade)
Targets Weak Spot #1 of EmptyDayConsistentPolicy:
  TransferPolicy._features uses only 5 linear variables and completely ignores
  solar hour-of-day (hod_sin, hod_cos), occupant heat load (occ_norm), and
  nonlinear thermal lag / cold-start dynamics.

Trained purely Supervised on GPU from thermal_residuals_*.parquet with strict
causality (no future 'power' leak!) and zero RLS double-counting.
"""
import math
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

_HERE = str(Path(__file__).resolve().parent)
if _HERE not in sys.path:
    sys.path.insert(0, _HERE)

from physics_foundation import EmptyDayConsistentPolicy, TransferPolicy, bounded_rls

THERMAL_FEAT_COLS = [
    'hod_sin', 'hod_cos', 'occ_norm',
    'oat_diff', 'slow_diff', 'group_diff', 'mode',
    'model_error', 'residual', 'theta_gain', 'log_area'
]
THERMAL_FEAT_DIM = len(THERMAL_FEAT_COLS)


def prepare_thermal_features_from_df(df):
    """Extract strictly causal [N, 11] float32 matrix and target [N, 1] from parquet df."""
    x = np.column_stack([
        df['hod_sin'].to_numpy(dtype=np.float32),
        df['hod_cos'].to_numpy(dtype=np.float32),
        df['occ_norm'].to_numpy(dtype=np.float32),
        np.clip(df['oat_diff'].to_numpy(dtype=np.float32), -5.0, 5.0),
        np.clip(df['slow_diff'].to_numpy(dtype=np.float32), -5.0, 5.0),
        np.clip(df['group_diff'].to_numpy(dtype=np.float32), -5.0, 5.0),
        df['mode'].to_numpy(dtype=np.float32),
        np.clip(df['model_error'].to_numpy(dtype=np.float32), 0.0, 2.0),
        np.clip(df['residual'].to_numpy(dtype=np.float32) / 2.0, -2.5, 2.5),
        np.log(np.clip(df['theta_gain'].to_numpy(dtype=np.float32), 1e-3, 10.0)),
        np.log1p(np.clip(df['zone_area'].to_numpy(dtype=np.float32), 0.0, 1e5)) / 5.0,
    ]).astype(np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=5.0, neginf=-5.0)
    y = np.clip(df['rls_err_K'].to_numpy(dtype=np.float32), -3.0, 3.0)[:, None]
    y = np.nan_to_num(y, nan=0.0)
    return x, y


class ThermalResidualResNet(nn.Module):
    """Per-zone Physics-Informed Residual Drift predictor (bounded output [-max_corr, max_corr])."""
    def __init__(self, in_dim=THERMAL_FEAT_DIM, hidden=192, max_corr=0.80):
        super().__init__()
        self.max_corr = float(max_corr)
        self.in_proj = nn.Sequential(nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.GELU())
        self.block1 = nn.Sequential(
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden),
        )
        self.block2 = nn.Sequential(
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden),
        )
        self.head = nn.Linear(hidden, 1)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        h = self.in_proj(x)
        h = F.gelu(h + self.block1(h))
        h = F.gelu(h + self.block2(h))
        return self.max_corr * torch.tanh(self.head(h))


class ThermalAugmentedPolicy(EmptyDayConsistentPolicy):
    """EmptyDayConsistentPolicy upgraded with GPU-trained Neural Thermal Drift residual."""
    name = 'empty_day_thermal_nn'

    def __init__(self, building_meta_data, action_spec=None, parameters=None,
                 thermal_nets=None, alpha_nn=1.0, dev=None):
        self._thermal_nets = thermal_nets if isinstance(thermal_nets, (list, tuple)) else (
            [thermal_nets] if thermal_nets is not None else [])
        self._alpha_nn = float(alpha_nn)
        self._dev = dev or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        super().__init__(building_meta_data, action_spec, parameters)

    def reset(self):
        super().reset()
        self._cur_ctx = (0.0, 1.0, 0.0)
        self._prev_ctx = None
        self._nn_corr_log = []

    def _extract_ctx(self, obs):
        hour = self._hour(obs)
        s_idx = self.field_index.get('hod_sin_000')
        c_idx = self.field_index.get('hod_cos_000')
        h_sin = float(obs[s_idx]) if s_idx is not None else math.sin(hour * math.pi / 12.0)
        h_cos = float(obs[c_idx]) if c_idx is not None else math.cos(hour * math.pi / 12.0)
        occ_raw = float(obs[self.occupancy_index]) if self.occupancy_index is not None else 0.0
        occ_norm = math.log1p(max(occ_raw, 0.0)) / 5.0 if np.isfinite(occ_raw) else 0.0
        return (h_sin, h_cos, occ_norm)

    def _predict_nn_drift(self, temperature, outside, mode, slow, ctx):
        if not self._thermal_nets or self._alpha_nn <= 0.0 or ctx is None:
            return np.zeros(self.n_zones, dtype=float)
        h_sin, h_cos, occ_norm = ctx
        feats = self._features(temperature, outside, mode, slow)
        x_np = np.column_stack([
            np.full(self.n_zones, h_sin, dtype=np.float32),
            np.full(self.n_zones, h_cos, dtype=np.float32),
            np.full(self.n_zones, occ_norm, dtype=np.float32),
            np.clip(feats[:, 1], -5.0, 5.0).astype(np.float32),
            np.clip(feats[:, 2], -5.0, 5.0).astype(np.float32),
            np.clip(feats[:, 3], -5.0, 5.0).astype(np.float32),
            feats[:, 4].astype(np.float32),
            np.clip(self.model_error, 0.0, 2.0).astype(np.float32),
            np.clip(self.residual / 2.0, -2.5, 2.5).astype(np.float32),
            np.log(np.clip(self.theta[:, 5], 1e-3, 10.0)).astype(np.float32),
            np.log1p(np.clip(self.zone_area, 0.0, 1e5)).astype(np.float32) / 5.0,
        ])
        x_np = np.nan_to_num(x_np, nan=0.0, posinf=5.0, neginf=-5.0)
        xt = torch.tensor(x_np, dtype=torch.float32, device=self._dev)
        with torch.no_grad():
            preds = [net(xt).squeeze(-1).cpu().numpy() for net in self._thermal_nets]
        delta_nn = self._alpha_nn * np.mean(preds, axis=0)
        return np.clip(delta_nn, -0.8, 0.8)

    def _learn(self, observed):
        delay = int(self.parameters['observation_delay'])
        if self.previous_observed is None or len(self.history) <= delay:
            return
        applied = list(self.history)[-(delay + 1)]
        old = self.previous_observed
        nn_corr = self._predict_nn_drift(old, applied['outside'], applied['mode'], applied['slow'], self._prev_ctx)

        # Causal model error update BEFORE RLS fit (including NN correction)
        features = self._features(old, applied['outside'], applied['mode'], applied['slow'])
        drift = np.einsum('ij,ij->i', self.theta[:, :5], features) + self.residual + nn_corr
        gain = self.theta[:, 5] / 1000.0
        prediction = (old + drift + gain * (applied['conductance'] * applied['supply'] + applied['reheat'])) / (
            1.0 + gain * applied['conductance'])
        error = np.where(np.isfinite(observed - prediction), np.minimum(np.abs(observed - prediction), 2.0), 2.0)
        previous_error = self.model_error.copy()

        # RLS fits strictly on residual NOT already explained by NN (zero double-counting!)
        power = (applied['conductance'] * (applied['supply'] - observed) + applied['reheat']) / 1000.0
        x = np.column_stack((features, power))
        delta = observed - old
        valid = np.isfinite(x).all(axis=1) & np.isfinite(delta) & (np.abs(delta) <= 3.0)
        lower = np.zeros_like(self.theta)
        upper = np.ones_like(self.theta) * np.array([2.0, 5.0, 2.0, 2.0, 2.0, 1.0])
        lower[:, 0] = -2.0
        lower[:, 5] = 0.5 * self._thermal_prior
        upper[:, 5] = 1.5 * self._thermal_prior
        if np.any(valid):
            fit, cov = bounded_rls(
                self.theta[valid], self.covariance[valid], x[valid],
                delta[valid] - self.residual[valid] - nn_corr[valid],
                lower[valid], upper[valid], float(self.parameters['forgetting_factor']))
            self.theta[valid], self.covariance[valid] = fit, cov
        remaining = np.clip(delta - (np.einsum('ni,ni->n', self.theta, x) + nn_corr), -5.0, 5.0)
        rate = float(self.parameters['disturbance_rate'])
        self.residual = np.clip((1.0 - rate) * self.residual + rate * remaining, -5.0, 5.0)
        self.model_error = 0.94 * previous_error + 0.06 * error

    def _free_drift(self, temperature, outside, mode):
        base_drift = super()._free_drift(temperature, outside, mode)
        nn_corr = self._predict_nn_drift(temperature, outside, mode, self.slow_temperature, self._cur_ctx)
        self._nn_corr_log.append(float(np.mean(np.abs(nn_corr))))
        return np.clip(base_drift + nn_corr, -5.0, 5.0)

    def get_action(self, obs):
        obs_arr = np.asarray(obs, dtype=float).reshape(-1)
        self._cur_ctx = self._extract_ctx(obs_arr)
        act = super().get_action(obs_arr)
        self._prev_ctx = self._cur_ctx
        return act


In [ ]:
%%writefile /kaggle/working/key_snapshot_profiler.py
"""Key-Position Counterfactual Snapshot Profiler.

Captures exact simulator state snapshots at pivotal operational moments across a 72h
building run (morning warm-up, peak cooling, mild shoulder economizer, comfort crisis,
empty-day relaxation, 72h weekend boiler hold) and evaluates counterfactual decision
arms from the exact same physical state + RNG seed to reveal AT WHICH POSITION,
WHICH DECISION IS STRONGEST.
"""
import copy
import math
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

_HERE = str(Path(__file__).resolve().parent)
if _HERE not in sys.path:
    sys.path.insert(0, _HERE)

from physics_foundation import EmptyDayConsistentPolicy
from warp_resip import AHU_ACT_DIM, WarpResiPPolicy

try:
    from thermal_residual_net import ThermalAugmentedPolicy
except Exception:
    ThermalAugmentedPolicy = None

try:
    from smart_buildings.smart_control.simulator.stochastic_convection_simulator import _seed_kernel_rng
except Exception:
    def _seed_kernel_rng(seed):
        return None


KEY_POSITIONS = [
    'POS_1_MORNING_WARMUP',
    'POS_2_PEAK_COOLING',
    'POS_3_MILD_SHOULDER',
    'POS_4_COMFORT_CRISIS',
    'POS_5_EMPTY_DAY_START',
    'POS_6_WEEKEND_WATER_HOLD',
]


def _make_modifier(kind):
    def _mod(step_cnt, z_f, g_f, xb):
        x = xb.copy()
        ahu_active = x[1] > -0.99
        if kind == 'sat_-2K' and ahu_active:
            x[0] = float(np.clip(x[0] - 0.20, -1.0, 1.0))
        elif kind == 'sat_+2K' and ahu_active:
            x[0] = float(np.clip(x[0] + 0.20, -1.0, 1.0))
        elif kind == 'press_-25%' and ahu_active:
            x[1] = float(np.clip(x[1] - 0.25, -1.0, 1.0))
        elif kind == 'press_+25%' and ahu_active:
            x[1] = float(np.clip(x[1] + 0.25, -1.0, 1.0))
        elif kind == 'recirc_eco' and ahu_active:
            x[2] = float(np.clip(x[2] - 0.40, -1.0, 1.0))
        elif kind == 'recirc_max' and ahu_active:
            x[2] = 0.40  # recirc = 0.70
        elif kind == 'water_325K':
            x[3] = -0.25
        elif kind == 'water_310K':
            x[3] = -1.00
        return x
    return _mod


def build_default_arms(thermal_nets=None, alpha_nn=0.5, iql_actors=None, dev=None):
    """Build standard physical perturbation arms plus optional trained Stage 1 / Stage 2 models."""
    arms = {
        'base': {'type': 'warp', 'modifier': None, 'actors': []},
        'sat_-2K': {'type': 'warp', 'modifier': _make_modifier('sat_-2K'), 'actors': []},
        'sat_+2K': {'type': 'warp', 'modifier': _make_modifier('sat_+2K'), 'actors': []},
        'press_-25%': {'type': 'warp', 'modifier': _make_modifier('press_-25%'), 'actors': []},
        'press_+25%': {'type': 'warp', 'modifier': _make_modifier('press_+25%'), 'actors': []},
        'recirc_eco': {'type': 'warp', 'modifier': _make_modifier('recirc_eco'), 'actors': []},
        'recirc_max': {'type': 'warp', 'modifier': _make_modifier('recirc_max'), 'actors': []},
        'water_325K': {'type': 'warp', 'modifier': _make_modifier('water_325K'), 'actors': []},
        'water_310K': {'type': 'warp', 'modifier': _make_modifier('water_310K'), 'actors': []},
    }
    if thermal_nets and ThermalAugmentedPolicy is not None:
        arms['stage1_thermal'] = {'type': 'thermal', 'nets': thermal_nets, 'alpha_nn': alpha_nn, 'dev': dev}
    if iql_actors:
        arms['stage2_iql_shield'] = {'type': 'warp', 'modifier': None, 'actors': iql_actors, 'dev': dev}
    return arms


def _copy_policy_state(src, dst, obs_arr):
    for attr in [
        'step_count', 'last_heating_step', 'empty_day_relaxed_steps', '_oat_ema',
    ]:
        if hasattr(src, attr):
            setattr(dst, attr, copy.deepcopy(getattr(src, attr)))
    for arr_attr in [
        'previous_observed', 'slow_temperature', 'residual', 'model_error',
        'theta', 'covariance', '_prev_delta_x',
    ]:
        if hasattr(src, arr_attr) and hasattr(dst, arr_attr):
            val = getattr(src, arr_attr)
            setattr(dst, arr_attr, None if val is None else np.array(val, copy=True))
    dst.history = copy.deepcopy(src.history)
    dst.last_action = copy.deepcopy(src.last_action)
    if hasattr(dst, '_extract_ctx'):
        dst._cur_ctx = dst._extract_ctx(obs_arr)
        dst._prev_ctx = dst._cur_ctx


def _step_components(env):
    try:
        rr = env.reward_function.compute_reward(env.building.reward_info)
        return (
            float(rr.normalized_productivity_regret),
            float(rr.normalized_energy_cost),
            float(rr.normalized_carbon_emission),
        )
    except Exception:
        return 0.0, 0.0, 0.0


def _instantiate_arm_policy(arm_spec, meta, action_spec, src_policy, obs_arr):
    if arm_spec['type'] == 'thermal' and ThermalAugmentedPolicy is not None:
        pol = ThermalAugmentedPolicy(
            meta, action_spec, thermal_nets=arm_spec['nets'],
            alpha_nn=arm_spec['alpha_nn'], dev=arm_spec.get('dev'),
        )
    else:
        pol = WarpResiPPolicy(
            meta, action_spec,
            actors=arm_spec.get('actors', []),
            action_modifier=arm_spec.get('modifier', None),
            dev=arm_spec.get('dev'),
            comfort_shield=True,
        )
    _copy_policy_state(src_policy, pol, obs_arr)
    return pol


def _run_branch_from_snapshot(snap, arm_spec, horizon, load_env_fn, create_meta_fn, sc_path, branch_seed):
    random.setstate(snap['py_rng'])
    np.random.set_state(snap['np_rng'])
    _seed_kernel_rng(branch_seed)

    clone_mode = 'deepcopy'
    try:
        env_b = copy.deepcopy(snap['env'])
        obs_b = np.array(snap['obs'], copy=True)
    except Exception:
        clone_mode = 'replay'
        env_b = load_env_fn(sc_path / 'config.gin')
        obs_b, _ = env_b.reset()
        for act in snap['action_history']:
            obs_b, _, _, _, _ = env_b.step(act)
        obs_b = np.asarray(obs_b, dtype=float).reshape(-1)
        _seed_kernel_rng(branch_seed)

    meta_b = create_meta_fn(env_b, sc_path)
    pol_b = _instantiate_arm_policy(arm_spec, meta_b, env_b.action_spec(), snap['policy'], obs_b)

    tot_r, tot_reg, tot_en, tot_cb = 0.0, 0.0, 0.0, 0.0
    for _ in range(horizon):
        act = pol_b.get_action(obs_b)
        obs_b, r, _, _, _ = env_b.step(act)
        obs_b = np.asarray(obs_b, dtype=float).reshape(-1)
        reg, en, cb = _step_components(env_b)
        tot_r += float(r)
        tot_reg += reg
        tot_en += en
        tot_cb += cb
    return {
        'total_rew': tot_r,
        'regret': tot_reg,
        'energy': tot_en,
        'carbon': tot_cb,
        'clone_mode': clone_mode,
    }


def collect_scenario_snapshots(sc_path, load_env_fn, create_meta_fn, steps=864, horizon=24, warmup=24):
    """Roll out Base once cleanly and snapshot env+policy state at up to 6 key positions."""
    env = load_env_fn(sc_path / 'config.gin')
    obs, _ = env.reset()
    meta = create_meta_fn(env, sc_path)
    pol = WarpResiPPolicy(meta, env.action_spec(), actors=[])

    captured = {}
    action_history = []
    base_rew_trace = []
    prev_ahu_run = False

    for st in range(steps):
        obs_arr = np.asarray(obs, dtype=float).reshape(-1)
        hour = pol._hour(obs_arr)
        act_pre = pol.get_action(obs_arr)
        _, g_f, xb, _ = pol._last_transition
        ahu_run = bool(np.any(act_pre['discrete_action'][:-1] == 1))
        water_target = 330.0 + 20.0 * float(xb[3])

        obs_next, r, _, _, _ = env.step(act_pre)
        reg, _, _ = _step_components(env)
        base_rew_trace.append(float(r))
        action_history.append(copy.deepcopy(act_pre))

        next_st = st + 1
        if warmup <= next_st <= steps - horizon - 1:
            obs_n_arr = np.asarray(obs_next, dtype=float).reshape(-1)
            next_hour = pol._hour(obs_n_arr)
            c_now_n = bool(obs_n_arr[pol.comfort_now_index] > 0.5) if pol.comfort_now_index is not None else bool(g_f[4] > 0.5)
            c_soon_n = bool(obs_n_arr[pol.comfort_soon_index] > 0.5) if pol.comfort_soon_index is not None else bool(g_f[5] > 0.5)
            oat_K_n = float(np.mean([
                pol._temperature(obs_n_arr, idx, 'outside_air_temperature_sensor', 290.0)
                for idx in pol.outside_obs
            ]))
            oat_C_n = oat_K_n - 273.15
            occ_raw_n = float(obs_n_arr[pol.occupancy_index]) if pol.occupancy_index is not None else 0.0
            occ_norm_n = math.log1p(max(occ_raw_n, 0.0)) / 5.0 if np.isfinite(occ_raw_n) else 0.0
            mean_T_n = float(np.mean(pol._zone_temperatures(obs_n_arr)))

            triggers = []
            if 'POS_1_MORNING_WARMUP' not in captured and (
                (not c_now_n and c_soon_n) or ((not prev_ahu_run) and ahu_run and 4.0 <= hour <= 10.5)
            ):
                triggers.append('POS_1_MORNING_WARMUP')
            if 'POS_2_PEAK_COOLING' not in captured and (
                c_now_n and occ_norm_n > 0.05 and ahu_run and 11.0 <= next_hour <= 16.5 and oat_C_n >= 18.0
            ):
                triggers.append('POS_2_PEAK_COOLING')
            if 'POS_3_MILD_SHOULDER' not in captured and (
                c_now_n and ahu_run and (mean_T_n - oat_K_n >= 1.5)
            ):
                triggers.append('POS_3_MILD_SHOULDER')
            if 'POS_4_COMFORT_CRISIS' not in captured and reg > 5e-4:
                triggers.append('POS_4_COMFORT_CRISIS')
            if 'POS_5_EMPTY_DAY_START' not in captured and getattr(pol, 'empty_day_relaxed_steps', 0) == 1:
                triggers.append('POS_5_EMPTY_DAY_START')
            if 'POS_6_WEEKEND_WATER_HOLD' not in captured and (
                (not ahu_run) and water_target >= 345.0 and (st - pol.last_heating_step >= 72)
            ):
                triggers.append('POS_6_WEEKEND_WATER_HOLD')

            if triggers:
                snap_obj = {
                    'step': next_st,
                    'hour': float(next_hour),
                    'oat_C': float(oat_C_n),
                    'occ_norm': float(occ_norm_n),
                    'obs': np.array(obs_n_arr, copy=True),
                    'policy': copy.deepcopy(pol),
                    'env': None,
                    'action_history': copy.deepcopy(action_history),
                    'py_rng': random.getstate(),
                    'np_rng': np.random.get_state(),
                }
                try:
                    snap_obj['env'] = copy.deepcopy(env)
                except Exception:
                    snap_obj['env'] = None
                for pos in triggers:
                    captured[pos] = snap_obj

        prev_ahu_run = ahu_run
        obs = obs_next

    for pos, sn in captured.items():
        s0 = sn['step']
        sn['uninterrupted_base_rew'] = float(sum(base_rew_trace[s0: s0 + horizon]))
    return captured


def profile_key_snapshots(scenario_paths, load_env_fn, create_meta_fn,
                          steps=864, horizon=24, arms=None, arm_builder=None,
                          out_dir=None, seed=42):
    """Run branching counterfactual audit across all captured key positions."""
    if arms is None and arm_builder is None:
        arms = build_default_arms()

    rows = []
    parity_errs = []
    clone_modes = set()
    all_arm_names = []

    for sc_idx, sc_path in enumerate(scenario_paths):
        bldg = sc_path.parts[-2]
        sc_key = f'{bldg}/{sc_path.parts[-1]}'
        sc_arms = arm_builder(bldg) if arm_builder is not None else arms
        for k in sc_arms:
            if k not in all_arm_names:
                all_arm_names.append(k)

        captured = collect_scenario_snapshots(
            sc_path, load_env_fn, create_meta_fn, steps=steps, horizon=horizon
        )
        print(f'[{sc_idx+1}/{len(scenario_paths)}] {sc_key}: captured {len(captured)}/{len(KEY_POSITIONS)} key positions {list(captured.keys())}')

        for pos, snap in captured.items():
            b_seed = seed * 1000 + sc_idx * 31 + snap['step']
            base_res = _run_branch_from_snapshot(
                snap, sc_arms['base'], horizon, load_env_fn, create_meta_fn, sc_path, b_seed
            )
            clone_modes.add(base_res['clone_mode'])
            parity_errs.append(abs(base_res['total_rew'] - snap['uninterrupted_base_rew']))

            for arm_name, arm_spec in sc_arms.items():
                res = base_res if arm_name == 'base' else _run_branch_from_snapshot(
                    snap, arm_spec, horizon, load_env_fn, create_meta_fn, sc_path, b_seed
                )
                rows.append({
                    'building': bldg,
                    'scenario': sc_key,
                    'position': pos,
                    'step': snap['step'],
                    'hour': round(snap['hour'], 2),
                    'oat_C': round(snap['oat_C'], 2),
                    'occ_norm': round(snap['occ_norm'], 3),
                    'arm': arm_name,
                    'arm_rew': res['total_rew'],
                    'base_rew': base_res['total_rew'],
                    'gain_total': res['total_rew'] - base_res['total_rew'],
                    'comfort_saved': base_res['regret'] - res['regret'],
                    'energy_saved': base_res['energy'] - res['energy'],
                    'carbon_saved': base_res['carbon'] - res['carbon'],
                })

    df_detail = pd.DataFrame(rows)
    if df_detail.empty:
        print('Khong thu thap duoc snapshot nao!')
        return df_detail, pd.DataFrame()

    max_parity = float(max(parity_errs)) if parity_errs else 0.0
    print(f'\nSnapshot Fidelity Check ({clone_modes}): max |cloned_base - orig_base| = {max_parity:.6f}')

    mat_gain = df_detail.pivot_table(index='position', columns='arm', values='gain_total', aggfunc='mean')
    arm_cols = [c for c in all_arm_names if c in mat_gain.columns and c != 'base']
    mat_gain = mat_gain.reindex(index=[p for p in KEY_POSITIONS if p in mat_gain.index], columns=arm_cols)

    df_summary = df_detail[df_detail['arm'] != 'base'].groupby(['position', 'arm']).agg(
        n=('gain_total', 'count'),
        gain_mean=('gain_total', 'mean'),
        win_pct=('gain_total', lambda s: float((s > 1e-6).mean() * 100.0)),
        comfort_saved=('comfort_saved', 'mean'),
        energy_saved=('energy_saved', 'mean'),
        carbon_saved=('carbon_saved', 'mean'),
    ).reset_index()

    print('\n=== MA TRAN QUYET DINH TAI CAC VI TRI QUAN TRONG (Mean 2h Gain vs Base) ===')
    print(mat_gain.round(5).to_string())

    print('\n=== TOP QUYET DINH MANH NHAT CHO TUNG VI TRI ===')
    for pos in mat_gain.index:
        sub = df_summary[df_summary['position'] == pos].sort_values('gain_mean', ascending=False)
        b = sub.iloc[0]
        w = sub.iloc[-1]
        print(
            f'  {pos:25s} (n={int(b["n"])}): MANH NHAT = {b["arm"]:16s} '
            f'(gain={b["gain_mean"]:+.5f}, win={b["win_pct"]:5.1f}%, cmf={b["comfort_saved"]:+.5f}, eng={b["energy_saved"]:+.5f}) '
            f'| TE NHAT = {w["arm"]} ({w["gain_mean"]:+.5f})'
        )

    if out_dir is not None:
        out_p = Path(out_dir)
        out_p.mkdir(parents=True, exist_ok=True)
        df_detail.to_csv(out_p / 'key_snapshots_detail.csv', index=False)
        mat_gain.to_csv(out_p / 'key_snapshots_matrix.csv')
        df_summary.to_csv(out_p / 'key_snapshots_summary.csv', index=False)

        fig, ax = plt.subplots(figsize=(max(9, len(arm_cols) * 1.15), 5.2))
        vals = mat_gain.fillna(0.0).to_numpy()
        vmax = max(1e-4, float(np.percentile(np.abs(vals), 95)))
        im = ax.imshow(vals, cmap='RdYlGn', aspect='auto', vmin=-vmax, vmax=vmax)
        ax.set_xticks(np.arange(len(arm_cols)))
        ax.set_xticklabels(arm_cols, rotation=30, ha='right', fontsize=9)
        ax.set_yticks(np.arange(len(mat_gain.index)))
        ax.set_yticklabels(mat_gain.index, fontsize=9)
        for i in range(vals.shape[0]):
            for j in range(vals.shape[1]):
                v = mat_gain.iloc[i, j]
                txt = 'nan' if pd.isna(v) else f'{v:+.4f}'
                ax.text(j, i, txt, ha='center', va='center', fontsize=8, color='black')
        plt.colorbar(im, ax=ax, label='Mean 2h Reward Gain vs Base (>0 = Thang Base)')
        ax.set_title('Key-Position x Decision Gain Matrix (Exact State Branching)')
        plt.tight_layout()
        plt.savefig(out_p / 'key_snapshots_heatmap.png', dpi=150)
        plt.show()

    return df_detail, mat_gain


In [ ]:
# Cell 1: Setup & Tu dong nap Model OOF (neu co) + Chon Kich ban dai dien
import os, sys, json, re
from pathlib import Path
import numpy as np
import pandas as pd
import torch

# --- CAU HINH SNAPSHOT PROFILER ---
SCENARIOS_PER_BLDG = 2    # 2 kich ban/toa nha = 12 kich ban (~5-8 phut CPU); dat None de quet full 30
STEPS_SCAN = 864          # Quet tren chuoi 72h
HORIZON = 24              # Cua so danh gia nhanh sau moi snapshot (24 buoc = 2 gio)
SEED = 42

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Thiet bi: {device} | SCENARIOS_PER_BLDG: {SCENARIOS_PER_BLDG} | HORIZON: {HORIZON}')

candidates = [
    Path('/kaggle/input/competitions/neurips-2026-smart-buildings'),
    Path('/kaggle/input/neurips-2026-smart-buildings'),
]
DATA_ROOT = next((p for p in candidates if (p / 'smart_buildings').exists()), None)
if DATA_ROOT is None:
    for base in [Path.cwd()] + list(Path.cwd().parents):
        if (base / 'smart_buildings').exists():
            DATA_ROOT = base
            break
assert DATA_ROOT is not None, 'Khong tim thay smart_buildings package!'

OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
for p in [OUT_DIR, Path.cwd(), DATA_ROOT]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

os.chdir(DATA_ROOT)
TRAIN_DIR = DATA_ROOT / 'smart_buildings' / 'training_scenarios'
from smart_buildings.evaluate import load_hybrid_environment, create_building_meta_data
from thermal_residual_net import ThermalResidualResNet
from warp_resip import DeepSetsWarpResiPActor
from key_snapshot_profiler import build_default_arms, profile_key_snapshots

ALL_SCENARIOS = sorted([p.parent for p in TRAIN_DIR.glob('*/*/config.gin')])
if SCENARIOS_PER_BLDG is None:
    SCAN_SCENARIOS = ALL_SCENARIOS
else:
    by_bldg = {}
    for sc in ALL_SCENARIOS:
        by_bldg.setdefault(sc.parts[-2], []).append(sc)
    SCAN_SCENARIOS = []
    for b in sorted(by_bldg):
        scs = by_bldg[b]
        # Chon regime dau & cuoi de da dang thoi tiet
        idxs = sorted(set(np.linspace(0, len(scs) - 1, min(SCENARIOS_PER_BLDG, len(scs)), dtype=int)))
        SCAN_SCENARIOS.extend([scs[i] for i in idxs])

def _find_latest(pattern):
    hits = []
    for d in [OUT_DIR, Path('/kaggle/input'), Path.cwd()]:
        if d.exists():
            hits.extend(list(d.rglob(pattern)))
    hits = sorted(hits)
    return hits[-1] if hits else None

f3_path = _find_latest('balanced_3fold_*.json')
if f3_path is not None:
    FOLDS_3 = json.loads(f3_path.read_text())
else:
    bldgs = sorted({p.parts[-2] for p in ALL_SCENARIOS})
    FOLDS_3 = [bldgs[i::3] for i in range(3)]
BLDG_TO_FOLD = {b: f_i for f_i, bs in enumerate(FOLDS_3) for b in bs}

# Tu dong nap checkpoint Stage 1 & Stage 2 (neu duoc attach vao Input) theo dung Fold (OOF)
s1_cfg_p = _find_latest('stage1_config.json')
alpha_nn = float(json.loads(s1_cfg_p.read_text()).get('best_alpha', 0.5)) if s1_cfg_p else 0.5

def _collect_fold_ckpts(pattern):
    out = {}
    for d in [OUT_DIR, Path('/kaggle/input')]:
        if not d.exists():
            continue
        for p in sorted(d.rglob(pattern)):
            if '_raw_trained' in p.name:
                continue
            m = re.search(r'fold(\d+)', p.name)
            if m:
                out[int(m.group(1))] = p
    return out

s1_ckpts = _collect_fold_ckpts('thermal_net_fold*.pt')
s2_ckpts = _collect_fold_ckpts('warp_resip_*_fold*.pt')
print(f'Kich ban quet: {len(SCAN_SCENARIOS)} | Balanced 3-Fold: {FOLDS_3}')
print(f'Tim thay checkpoint Stage 1: {sorted(s1_ckpts)} (alpha={alpha_nn}) | Stage 2: {sorted(s2_ckpts)}')

_loaded_s1, _loaded_s2 = {}, {}
for f_i, p in s1_ckpts.items():
    net = ThermalResidualResNet().to(device)
    net.load_state_dict(torch.load(p, map_location=device, weights_only=True))
    net.eval()
    _loaded_s1[f_i] = net
for f_i, p in s2_ckpts.items():
    act = DeepSetsWarpResiPActor().to(device)
    act.load_state_dict(torch.load(p, map_location=device, weights_only=True))
    act.eval()
    _loaded_s2[f_i] = act

def arm_builder(bldg):
    f_i = BLDG_TO_FOLD.get(bldg, -1)
    t_nets = [_loaded_s1[f_i]] if f_i in _loaded_s1 else None
    i_acts = [_loaded_s2[f_i]] if f_i in _loaded_s2 else None
    return build_default_arms(thermal_nets=t_nets, alpha_nn=alpha_nn, iql_actors=i_acts, dev=device)


In [ ]:
# Cell 2: Chup Snapshot & Phan tich Ma tran (Vi tri Quan trong x Quyet dinh)
df_detail, mat_gain = profile_key_snapshots(
    SCAN_SCENARIOS,
    load_hybrid_environment,
    create_building_meta_data,
    steps=STEPS_SCAN,
    horizon=HORIZON,
    arm_builder=arm_builder,
    out_dir=OUT_DIR,
    seed=SEED,
)

if not df_detail.empty:
    print('\n=== TONG HOP THEO TOA NHA x VI TRI: QUYET DINH MANH NHAT ===')
    sub = df_detail[df_detail['arm'] != 'base']
    g = sub.groupby(['building', 'position', 'arm'])['gain_total'].mean().reset_index()
    idx_best = g.groupby(['building', 'position'])['gain_total'].idxmax()
    best_bldg = g.loc[idx_best].sort_values(['position', 'building'])
    print(best_bldg.to_string(index=False))
